<p class="lead">This notebook is the R edition of the eighth unit of the course, in which the multilayer perceptron of Unit 6 meets the data that made neural networks famous in medicine: images. The lecture slides are a map; this text is the territory. Every figure and number below is produced by R code you can run and change. Sections 4 and 5 train perceptrons on images, measure them with the methodology of Supplement S1, and vary their topology — depth, width, activation, regularisation, optimiser — in a controlled design that shows what improves classification and what only appears to. Sections 6 and 7 ask <em>why</em> a perceptron is an awkward image classifier, build the convolution operation by hand, and reproduce on simulated radiographs the most-documented failure of medical imaging AI: a network that learns the hospital instead of the disease.</p>

<div class="box key"><h4>Learning objectives</h4>
<ul class="objectives">
<li>Explain what a pixel value means in each major imaging modality, represent an image as a matrix and as a feature vector, and describe the pattern-recognition pipeline of Unit 1 as it applies to images.</li>
<li>Apply and justify the standard pre-processing steps — intensity normalisation, histogram equalisation, resizing, Gaussian denoising and data augmentation — and state which of them preserve the class label and which do not.</li>
<li>Compute first-order, texture (grey-level co-occurrence) and shape features from an image, trace the co-occurrence matrix by hand, and train the classifiers of Units 3–4 on them as a classical baseline.</li>
<li>Train a multilayer perceptron on flattened images, evaluate it with repeated stratified cross-validation, read its confusion matrix, visualise its first-layer weights, and choose a decision threshold for a screening setting.</li>
<li>Design and report an honest experiment on network topology: a factorial grid evaluated with repeated cross-validation, variance across folds, learning curves, and nested cross-validation for the final estimate.</li>
<li>Explain translation invariance, trace a two-dimensional convolution and a max-pooling step by hand, implement both in R, and show why a convolutional front end is the bridge from the perceptron to deep learning.</li>
<li>Reproduce shortcut learning on image data, build an occlusion-sensitivity map that reveals it, and describe class imbalance, label noise, external validation, reader studies and the regulatory status of imaging AI.</li>
<li>Read the landmark papers of medical imaging AI critically: identify the sample, the reference standard, the metric, the comparator, and what is missing.</li>
</ul></div>

<div class="box bridge"><h4>How to read this notebook</h4>
<p>Read it linearly once, executing each chunk as you reach it; then return to the experiments of Sections 5–7 and change one factor at a time. Exercises (green boxes) ask you to modify the chunk above them. Blue boxes carry key ideas, orange boxes carry common misconceptions, and purple boxes connect this unit to other units or to the bioinformatics course (BGM324). The notebook uses only base R and the standard course packages — <code>nnet</code> and <code>neuralnet</code> for the perceptron,<sup class="c"><a href="#r35">[35]</a></sup><sup class="c"><a href="#r36">[36]</a></sup> <code>e1071</code>, <code>randomForest</code> and <code>class</code> for the classical baseline,<sup class="c"><a href="#r37">[37]</a></sup><sup class="c"><a href="#r38">[38]</a></sup> and <code>pROC</code> for the ROC curve — with no deep-learning framework, because the aim is to understand the operations, not to run a library. In a fresh Google Colab R runtime a student runs once <code>install.packages(c("nnet","neuralnet","e1071","randomForest","pROC"))</code>. Two image sources are used throughout, both <strong>simulated</strong>: a set of 8 × 8 synthetic glyphs (six geometric shapes) that plays the role of the handwritten-digit benchmark used in the Python edition,<sup class="c"><a href="#r10">[10]</a></sup> and a generator of 16 × 16 radiograph-like patches written in Section 1.3. R has no bundled digit or medical-image dataset, so these replace them; the images are built to control the experiment — the position, size and contrast of every structure are known because we chose them — and they are not real radiographs. Section 1.3 lists the real datasets you should know.</p></div>



In [ ]:
# Libraries used throughout. Seeds are fixed so that every number quoted in the text is reproducible.
suppressPackageStartupMessages({
  library(nnet); library(neuralnet); library(e1071); library(class)
  library(randomForest); library(pROC); library(MASS)
})
set.seed(0)
palette_pos <- "#d2691e"; palette_neg <- "#2b5d9b"; palette_acc <- "#3a7d44"
cat(R.version.string, "| nnet", as.character(packageVersion("nnet")),
    "| neuralnet", as.character(packageVersion("neuralnet")), "\n")



The printout records the R and package versions these notes were built with, the R analogue of printing library versions at the top of a Python notebook. `set.seed(0)` makes every random split and every simulated image below reproducible.

## 1. Medical images as patterns

### 1.1 What a pixel means

A medical image is a measurement, and a classifier that reads it inherits the physics of the measurement. The survey by Litjens and colleagues, which catalogued more than 300 deep-learning papers on medical images by 2017, organises the field by modality for exactly this reason: what a network can learn depends on what the numbers in the image encode.<sup class="c"><a href="#r1">[1]</a></sup> The table below summarises the modalities you will meet in the literature of this unit.

| Modality | What a pixel (or voxel) value encodes | Typical size and bit depth | Classification tasks in the literature |
|---|---|---|---|
| **Projection radiography** (X-ray) | Integrated attenuation of X-rays along the beam path; dense tissue (bone, metal) attenuates more and appears bright by convention | 2,000–3,000 px per side; 12–16 bits | pneumonia, nodules, fractures, 14 thoracic labels of ChestX-ray14 |
| **Computed tomography** (CT) | Linear attenuation coefficient \( \mu \) of a small volume, reported in **Hounsfield units** \( \mathrm{HU} = 1000\,(\mu - \mu_{\text{water}})/(\mu_{\text{water}} - \mu_{\text{air}}) \), so water is 0, air about −1000 and cortical bone about +1000<sup class="c"><a href="#r2">[2]</a></sup> | 512 × 512 per slice, hundreds of slices; 12 bits | lung nodules, haemorrhage, COVID-19 pneumonia, organ segmentation |
| **Magnetic resonance** (MRI) | Signal of hydrogen nuclei after radio-frequency excitation; depends on proton density and relaxation times (T1, T2) and on the pulse sequence — **arbitrary units**, not comparable across scanners without normalisation | 256 × 256 per slice; 12–16 bits | brain tumour grading, multiple sclerosis lesions, prostate cancer |
| **Ultrasound** | Amplitude of the echo returned from acoustic-impedance boundaries, with speckle noise intrinsic to the method | 500–1,000 px per side; 8 bits | breast lesions, thyroid nodules, fetal measurements |
| **Histopathology** (whole-slide images) | Colour of stained tissue under transmitted light; three channels (RGB); the stain (H&E) is a chemical, not a physical, contrast | 100,000 px per side, processed as tiles; 8 bits per channel | tumour detection, grading, mitosis counting |
| **Dermoscopy** | Reflected light from the skin surface under magnification and polarisation; RGB | 1,000–6,000 px per side; 8 bits per channel | melanoma versus benign naevus (ISIC) |
| **Retinal photography** (fundus) | Reflected light from the retina through the pupil; RGB | 2,000–4,000 px per side; 8 bits per channel | diabetic retinopathy grading, glaucoma |

Three consequences follow. CT values are physical quantities on a shared scale, so a threshold of −500 HU means the same thing on every scanner; MRI and ultrasound values are not, so **intensity normalisation** (Section 2.1) is mandatory before training on images from more than one machine. Clinical images have 12–16 bits of depth but are displayed through a **window** (a range of values mapped to the visible grey scale), so a model trained on windowed 8-bit images sees less than the workstation stores. And images are stored with their acquisition parameters in the **DICOM** standard (Digital Imaging and Communications in Medicine), whose header records scanner, protocol, pixel spacing and, unless stripped, the patient's identity: de-identification (Unit 1) is a precondition of every imaging dataset, and the fields that survive it — manufacturer, institution — are the very confounds that Section 7 shows a network can exploit.

### 1.2 Image = matrix = vector

An image with \( H \) rows and \( W \) columns is a matrix \( \mathbf{I} \in \mathbb{R}^{H \times W} \) whose entry \( I_{rc} \) is the intensity of the pixel at row \( r \) and column \( c \); a colour image adds a third index for the channel. Every classifier of Units 2–6 expects a *vector*, so the matrix is flattened into \( \mathbf{x} = (I_{11}, I_{12}, \dots, I_{HW}) \) of length \( p = H \cdot W \). The flattening is reversible and loses nothing — but it discards the *adjacency* of pixels: the classifier is not told that two pixels side by side in the image are neighbours while two that are one row apart are also neighbours. Section 6 shows that this is the central weakness of the perceptron as an image classifier. The first image source of the unit is a set of 8 × 8 synthetic glyphs generated below; the chunk prints one of them as the matrix it is.



In [ ]:
SG <- 8L                                   # glyph side (8 x 8, like the Python digit benchmark)
GLYPHS <- c("vbar","hbar","diagL","diagR","cross","ring")
draw_glyph <- function(cls){               # one noisy glyph as an SG x SG matrix in [0,1]
  img <- matrix(0, SG, SG); gy <- matrix(rep(1:SG, SG), SG, SG); gx <- t(gy)
  ctr <- SG/2 + 0.5; cx <- ctr + round(runif(1,-1,1)); cy <- ctr + round(runif(1,-1,1)); th <- runif(1,0.7,1.1)
  if (cls=="vbar")  img[abs(gx-cx) <= th] <- 1
  else if (cls=="hbar")  img[abs(gy-cy) <= th] <- 1
  else if (cls=="diagL") img[abs((gx-cx)-(gy-cy)) <= th*1.3] <- 1
  else if (cls=="diagR") img[abs((gx-cx)+(gy-cy)) <= th*1.3] <- 1
  else if (cls=="cross") { img[abs(gx-cx) <= th] <- 1; img[abs(gy-cy) <= th] <- 1 }
  else if (cls=="ring")  { rr <- sqrt((gx-cx)^2+(gy-cy)^2); img[abs(rr-2.3) <= th] <- 1 }
  pmin(pmax(img*runif(1,0.5,0.9) + matrix(rnorm(SG*SG,0,0.33),SG,SG), 0), 1)
}
set.seed(7); g1 <- draw_glyph("cross")
cat("A 'cross' glyph as an 8 x 8 matrix (0 = background, 1 = bright):\n"); print(round(g1,2))
cat("Flattened:", round(as.vector(g1)[1:10],2), "...  length", length(g1), "\n")
par(mfrow=c(1,2), mar=c(4,4,2,1))
image(t(g1[SG:1,]), col=grey.colors(32, start=1, end=0), axes=FALSE, main="'cross' as an 8 x 8 matrix"); box()
barplot(as.vector(g1), col=palette_neg, border=NA, space=0, xlab="vector index", ylab="intensity",
        main="the same matrix flattened"); abline(v=(1:7)*SG*1.2, col=palette_pos, lty=2, lwd=0.6)



<p class="caption"><b>Figure 1.</b> Left: a synthetic glyph as the 8 × 8 matrix of numbers that the object actually is; the picture is only a rendering of these numbers. Right: the 64-element vector obtained by reading the matrix column by column, the order R's <code>as.vector</code> uses (the dashed orange lines mark the column boundaries). To the perceptron, the last pixel of one column and the first of the next are as unrelated as any other pair, although in the image they may be vertical neighbours. The glyphs stand in for the handwritten-digit benchmark of the Python edition,<sup class="c"><a href="#r10">[10]</a></sup> which R does not bundle.</p>

### 1.3 Simulated radiograph patches, and the real datasets they stand in for

The glyphs are small images, but they are not medical, and a public course notebook cannot bundle radiographs. The second image source of the unit is therefore generated by a function written below. Each image is a 16 × 16 grey-level **patch** that imitates a small region of a radiograph: a smooth, cloud-like background texture (a sum of random fields at three spatial scales, in the manner of the *Perlin noise* used in computer graphics), a random overall exposure, and additive pixel noise. A positive patch additionally contains either a **nodule** — a brighter, roughly round blob of random position, radius and contrast — or a cluster of **microcalcifications**, a handful of very small bright points. The position of the lesion is drawn around the centre of the patch with a controllable **jitter**, because that is how patches reach a classifier in a computer-aided-detection system: a candidate detector proposes a location and a second-stage classifier decides whether the candidate is a lesion or a false positive, the design of the LUNA16 lung-nodule challenge.<sup class="c"><a href="#r11">[11]</a></sup> Finally, the generator can stamp a small bright rectangle in a corner of positive patches from a fictitious "hospital A" — the kind of laterality marker, ruler or annotation that real radiographs carry and that Section 7 shows a network will learn instead of the lesion.

<div class="box warn"><h4>These are simulations, not radiographs</h4><p>The patches below are generated by a dozen lines of code so that the experiment can be <em>controlled</em>: we know the position, size and contrast of every lesion, we can produce as many images as we need, and we can plant a confound and remove it. None of that is possible with real data, and nothing learned about <em>how well</em> a model classifies these patches transfers to radiographs. What transfers is the behaviour of the methods — what pre-processing does, why the perceptron is sensitive to position, how a shortcut is learned and revealed. The real datasets you should know, all public and all built with the labelling and de-identification effort described in their papers, are: <strong>ChestX-ray14</strong>, 112,120 frontal chest radiographs of 30,805 patients with 14 labels extracted from the reports by text mining;<sup class="c"><a href="#r3">[3]</a></sup> <strong>CheXpert</strong>, 224,316 radiographs of 65,240 patients with labels that record uncertainty;<sup class="c"><a href="#r4">[4]</a></sup> <strong>MIMIC-CXR</strong>, 377,110 images from 227,835 studies with their free-text reports;<sup class="c"><a href="#r5">[5]</a></sup> <strong>CBIS-DDSM</strong>, a curated and re-segmented subset of the Digital Database for Screening Mammography with verified pathology;<sup class="c"><a href="#r6">[6]</a></sup> <strong>HAM10000</strong>, 10,015 dermatoscopic images of pigmented lesions, the training set of the ISIC melanoma challenges;<sup class="c"><a href="#r7">[7]</a></sup> and <strong>MedMNIST v2</strong>, a collection of 18 small standardised datasets (28 × 28 images) that is the natural next step after this notebook.<sup class="c"><a href="#r8">[8]</a></sup> Remember also that the Wisconsin breast-cancer data used throughout the course are themselves <em>image-derived</em>: the 30 features were computed from digitised micrographs of cell nuclei by a semi-automatic segmentation program,<sup class="c"><a href="#r9">[9]</a></sup> which is the hand-crafted-feature pipeline of Section 3 applied in 1993.</p></div>



In [ ]:
# A reusable generator of simulated radiograph-like patches (NOT real images; see the box above).
SIZE <- 16L
up_mat <- function(size, sc){                       # separable bilinear-upsampling weight matrix
  xi <- seq(1, sc+1, length.out=size); x0 <- floor(xi); x1 <- pmin(x0+1, sc+1); fx <- xi-x0
  W <- matrix(0, size, sc+1)
  for (r in 1:size){ W[r,x0[r]] <- W[r,x0[r]]+(1-fx[r]); W[r,x1[r]] <- W[r,x1[r]]+fx[r] }; W
}
SCALES <- c(2,4,8); UM <- lapply(SCALES, function(sc) up_mat(SIZE, sc))
smooth_bg <- function(){                             # cloud-like texture: random grids at 3 scales, summed
  out <- matrix(0, SIZE, SIZE)
  for (k in seq_along(SCALES)){ sc <- SCALES[k]; g <- matrix(rnorm((sc+1)^2), sc+1, sc+1)
    out <- out + (1/sc) * (UM[[k]] %*% g %*% t(UM[[k]])) }
  out
}
make_patches <- function(n, lesion="nodule", jitter=3, hospital="B", marker_rate=0, positive=NULL,
                         contrast=c(0.3,0.6)){
  X <- array(0, c(n, SIZE, SIZE)); y <- integer(n)
  gy <- matrix(rep(1:SIZE, SIZE), SIZE, SIZE); gx <- t(gy)
  for (i in 1:n){
    img <- smooth_bg(); img <- (img-mean(img))/(sd(img)+1e-9)*0.10 + 0.45 + runif(1,-0.08,0.08)
    pos <- if (is.null(positive)) (runif(1)<0.5) else as.logical(positive)
    cx <- SIZE/2+0.5 + runif(1,-jitter,jitter); cy <- SIZE/2+0.5 + runif(1,-jitter,jitter)
    if (pos && lesion=="nodule"){ r <- runif(1,1.5,3); cc <- runif(1,contrast[1],contrast[2])
      img <- img + cc*exp(-((gx-cx)^2+(gy-cy)^2)/(2*(r/2)^2)) }
    else if (pos && lesion=="microcalc") for (j in 1:sample(3:6,1)){
      px <- min(max(round(cx+rnorm(1,0,1.3)),1),SIZE); py <- min(max(round(cy+rnorm(1,0,1.3)),1),SIZE)
      img[py,px] <- img[py,px] + runif(1,0.35,0.6) }
    if (hospital=="A" && pos && runif(1)<marker_rate) img[2:3,(SIZE-3):(SIZE-1)] <- 1.0   # corner marker
    img <- img + matrix(rnorm(SIZE*SIZE,0,0.03), SIZE, SIZE)
    X[i,,] <- pmin(pmax(img,0),1); y[i] <- as.integer(pos)
  }
  list(X=X, y=y)
}
set.seed(0);  nod <- make_patches(1000, "nodule")
set.seed(1);  mc  <- make_patches(1000, "microcalc")
cat(sprintf("nodule set: %d x %dx%d | positives %d | mean intensity neg %.3f, pos %.3f\n",
            dim(nod$X)[1], SIZE, SIZE, sum(nod$y), mean(nod$X[nod$y==0,,]), mean(nod$X[nod$y==1,,])))
cat(sprintf("microcalcification set: positives %d\n", sum(mc$y)))



The two printed mean intensities of the nodule classes differ only in the third decimal, which is deliberate: a classifier that looks at average brightness alone cannot solve this task, as Figure 2 and Section 3 confirm.



In [ ]:
set.seed(5); hA <- make_patches(6, "nodule", hospital="A", marker_rate=1.0)
show <- function(m, ...) image(t(m[SIZE:1,]), col=grey.colors(32,1,0), axes=FALSE, zlim=c(0,1), ...)
rows <- list("no lesion"=which(nod$y==0)[1:6], "nodule"=which(nod$y==1)[1:6],
             "microcalc."=which(mc$y==1)[1:6])
par(mfrow=c(4,6), mar=c(0.4,0.4,1.6,0.4), oma=c(0,7,2,0))
for (nm in names(rows)) for (j in seq_len(6)){ src <- if (nm=="microcalc.") mc$X else nod$X
  show(src[rows[[nm]][j],,]); if (j==1) mtext(nm, side=2, line=1, las=1, cex=0.7) }
for (j in 1:6){ show(hA$X[j,,]); if (j==1) mtext("hospital A\n(marker)", side=2, line=1, las=1, cex=0.7) }
mtext("Simulated 16 x 16 patches (the generator above; not real radiographs)", outer=TRUE, cex=0.9, font=2)



<p class="caption"><b>Figure 2.</b> Six examples of each kind of simulated patch. The nodules (second row) vary in position, size and contrast, and some are barely brighter than the background texture — the task is deliberately not trivial. The microcalcification clusters (third row) are small, bright and sparse, a pattern that first-order statistics capture poorly but local filters capture well (Section 6). The last row shows the fictitious hospital A, whose positive patches carry a bright rectangle in the top-right corner; everything else about them is generated by the same code. Because the two nodule classes have almost identical mean intensity, average brightness alone will not solve the task.</p>

### 1.4 The pipeline of Unit 1, applied to images

Unit 1 described every pattern-recognition system as sensing, pre-processing and segmentation, feature extraction, classification and decision. For images the stages are:

1. **Acquisition** — the physics of Section 1.1 produces a DICOM file; a model trained on one protocol is not validated for another.
2. **Pre-processing** (Section 2) — normalise intensities, resample to a common size, suppress noise and, during training, augment with label-preserving transformations.
3. **Segmentation** — isolate the structure of interest by thresholding and connected components (Section 3.3) or, in modern practice, by a network that labels every pixel (beyond this unit).
4. **Feature extraction** (Section 3) — compute a fixed-length vector: first-order statistics, texture, shape. In a neural network (Sections 4–6) this stage is *learned*.
5. **Classification and decision** (Sections 4–5, 7) — a classifier maps the vector to a probability, and a threshold chosen for the clinical setting maps the probability to an action.

The distinction between stages 4 and 5 organises the unit: Section 3 keeps them separate, as the field did until about 2012; Section 4 merges them in the perceptron; Section 6 shows that the merger works far better when the architecture respects the structure of images.

## 2. Pre-processing

Pre-processing transforms every image by the same rule before any learning happens, to remove variation that is *not* informative about the class — scanner gain, exposure, resolution, noise — so that the classifier's capacity is spent on variation that is. Each operation below is defined, implemented in a few lines of base R, and shown on a glyph and on a synthetic patch.

### 2.1 Intensity normalisation and histogram equalisation

Two linear normalisations are universal. **Min–max scaling** maps the image to \( [0,1] \): \( I' = (I - I_{\min})/(I_{\max} - I_{\min}) \); it removes offset and gain but is dominated by a single extreme pixel. **Standardisation** (z-scoring), \( I' = (I - \mu_I)/\sigma_I \), is the per-image analogue of the column standardisation you have used since Unit 3 and is the usual choice for MRI, where absolute values are meaningless. For CT, where values are physical, the standard operation is instead **windowing**: values below a lower bound are clipped to black, values above an upper bound to white, and the interval in between is stretched linearly; a "lung window" of about −1,000 to +400 HU shows the parenchyma, a "bone window" of about −500 to +1,300 HU shows the skeleton, and the same CT slice looks entirely different through the two.

**Histogram equalisation** is non-linear: it remaps intensities so that the histogram of the output is as flat as possible, which spreads the grey levels of a low-contrast image across the whole range. The algorithm is three lines. For an image with \( L \) grey levels, compute the histogram \( h(v) \), its cumulative distribution \( F(v) = \sum_{u \le v} h(u)/(HW) \), and map each pixel to \( v' = F(v) \). The chunk implements all three in base R and applies them to a glyph and to a low-contrast patch.



In [ ]:
minmax  <- function(m) (m - min(m)) / (max(m) - min(m) + 1e-12)
zscore  <- function(m) (m - mean(m)) / (sd(m) + 1e-12)
equalise <- function(m, levels=256){                    # histogram equalisation via the cumulative distribution
  q   <- pmin(round(minmax(m) * (levels-1)), levels-1)
  cdf <- cumsum(tabulate(q+1, levels)) / length(q)
  matrix(cdf[q+1], nrow(m), ncol(m))
}
low <- 0.25 + 0.25 * nod$X[which(nod$y==1)[4],,]          # a positive patch with contrast cut to a quarter
set.seed(7); dig <- draw_glyph("ring")
cat(sprintf("glyph: range [%.2f, %.2f] mean %.2f sd %.2f | after z-score: mean %.2f sd %.2f\n",
            min(dig), max(dig), mean(dig), sd(dig), mean(zscore(dig)), sd(zscore(dig))))
cat(sprintf("low-contrast patch: range [%.3f, %.3f] -> min-max [%.1f, %.1f] -> equalised sd %.3f (was %.3f)\n",
            min(low), max(low), min(minmax(low)), max(minmax(low)), sd(equalise(low)), sd(low)))
par(mfcol=c(2,4), mar=c(3,3,2,1))
for (pair in list(list(dig,"glyph"), list(low,"low-contrast patch"))){ im <- pair[[1]]; nm <- pair[[2]]
  image(t(im[nrow(im):1,]), col=grey.colors(32,1,0), axes=FALSE, main=paste(nm,"original")); box()
  image(t(minmax(im)[nrow(im):1,]), col=grey.colors(32,1,0), axes=FALSE, main="min-max to [0,1]"); box()
  image(t(equalise(im)[nrow(im):1,]), col=grey.colors(32,1,0), axes=FALSE, main="histogram-equalised"); box()
  hist(minmax(im), breaks=16, col=adjustcolor(palette_neg,0.6), border=NA, main="histograms", xlab="intensity")
  hist(equalise(im), breaks=16, col=adjustcolor(palette_pos,0.5), border=NA, add=TRUE) }



<p class="caption"><b>Figure 3.</b> Top-to-bottom within each pair: the original image, its min–max version, its equalised version, and the two histograms (blue min–max, orange equalised). Min–max scaling restores the display range but not the distribution: the low-contrast patch's histogram is still a narrow peak. Equalisation spreads the grey levels so that each is used about equally often, which makes the nodule visible and raises the standard deviation several-fold, as printed. On the glyph, where most pixels are background, equalisation mostly exaggerates the noise. Equalisation is a global operation; the contrast-limited adaptive version (CLAHE) applied in tiles is the one used in practice on radiographs and in the pre-processing of the retinal and dermoscopic datasets of Section 8.</p>

<div class="box warn"><h4>Normalise per image, or per dataset?</h4><p>Per-image normalisation (this section) removes exposure differences between images; per-feature standardisation across the training set puts every pixel position on the same scale for the optimiser. They are different operations and are usually both applied. Whatever is done, the parameters of any <em>dataset-level</em> normalisation — the column means and standard deviations — must be computed on the training folds only and applied to the test fold, exactly as Supplement S1 required for leakage. In this notebook every cross-validated model standardises inside the fold (the helper <code>sc_fit</code> of Section 3.4 recomputes the centre and scale on the training rows), which is the R equivalent of fitting a scaler on the training split alone.</p></div>

### 2.2 Resizing and denoising

Networks need inputs of a fixed size, and images arrive in many sizes and pixel spacings. **Resizing** by interpolation changes the number of pixels; the interpolation order matters — nearest-neighbour keeps blocks, bilinear produces smooth edges that were never measured. Reducing the size by averaging blocks of pixels (**downsampling**) is the operation that pooling layers perform in Section 6. **Denoising** with a Gaussian filter replaces each pixel by a weighted average of its neighbours with weights \( \propto \exp(-d^2/2\sigma^2) \); it is a convolution (Section 6.2), and its single parameter \( \sigma \) trades noise against sharpness. The chunk implements each operation in base R and applies them to a glyph and to a patch to which extra noise has been added.



In [ ]:
zoom_nn <- function(m, k){ H<-nrow(m); m[rep(1:H, each=k), rep(1:ncol(m), each=k)] }      # nearest-neighbour
zoom_bl <- function(m, size){ W <- up_mat(size, nrow(m)-1); W %*% m %*% t(W) }            # bilinear via up_mat
block_mean <- function(m, s){ H<-nrow(m)/s; W<-ncol(m)/s                                    # downsample by s x s mean
  outer(1:H,1:W, Vectorize(function(r,c) mean(m[((r-1)*s+1):(r*s),((c-1)*s+1):(c*s)]))) }
gkern <- function(sigma, sz){ r <- -(sz%/%2):(sz%/%2); g <- exp(-(outer(r^2,r^2,"+"))/(2*sigma^2)); g/sum(g) }
conv_same <- function(I, K){ k<-nrow(K); p<-k%/%2; H<-nrow(I); W<-ncol(I)                   # "same"-padded correlation
  Ip <- matrix(0, H+2*p, W+2*p); Ip[(p+1):(p+H),(p+1):(p+W)] <- I; O <- matrix(0,H,W)
  for (r in 1:H) for (c in 1:W) O[r,c] <- sum(K * Ip[r:(r+k-1), c:(c+k-1)]); O }
gauss_filter <- function(m, sigma) conv_same(m, gkern(sigma, 2*ceiling(2*sigma)+1))
set.seed(2); patch <- nod$X[which(nod$y==1)[5],,]; noisy <- pmin(pmax(patch + rnorm(SIZE^2,0,0.12),0),1)
mae <- function(a,b) mean(abs(a-b))
cat(sprintf("mean abs. error vs clean patch: noisy %.3f | sigma 0.5 %.3f | 1.0 %.3f | 2.0 %.3f\n",
            mae(noisy,patch), mae(gauss_filter(noisy,0.5),patch), mae(gauss_filter(noisy,1),patch),
            mae(gauss_filter(noisy,2),patch)))
par(mfrow=c(2,5), mar=c(1,1,2,1))
set.seed(7); gl <- draw_glyph("diagR")
for (im in list(list(gl,"glyph 8x8"), list(zoom_nn(gl,4),"zoom x4 nearest"), list(zoom_bl(gl,32),"zoom x4 bilinear"),
                list(block_mean(zoom_bl(gl,32),2),"down to 16x16"), list(block_mean(patch,2),"patch to 8x8")))
  { image(t(im[[1]][nrow(im[[1]]):1,]), col=grey.colors(32,1,0), axes=FALSE, main=im[[2]], cex.main=0.8); box() }
for (im in list(list(patch,"clean patch"), list(noisy,"patch + noise"), list(gauss_filter(noisy,0.5),"Gaussian s=0.5"),
                list(gauss_filter(noisy,1),"Gaussian s=1"), list(gauss_filter(noisy,2),"Gaussian s=2")))
  { image(t(im[[1]][nrow(im[[1]]):1,]), col=grey.colors(32,1,0), axes=FALSE, main=im[[2]], cex.main=0.8); box() }



<p class="caption"><b>Figure 4.</b> Top: a glyph enlarged four-fold by nearest-neighbour and by bilinear interpolation, then downsampled; the nearest version is honest about the eight-by-eight grid, the bilinear version invents smooth edges. Top right: a patch downsampled to 8 × 8 by averaging 2 × 2 blocks — a quarter of the pixels, the nodule still visible. Bottom: the clean patch, the same patch with heavy added noise, and the noisy patch smoothed with Gaussian filters of increasing width. The printed mean absolute errors against the clean patch fall from the noisy image to \( \sigma = 1 \) and rise again at \( \sigma = 2 \), where the filter begins to erase the nodule itself: denoising has an optimum, and the optimum depends on the size of the structure one wants to keep.</p>

### 2.3 Data augmentation

A classifier that sees each image once, in one orientation, learns the orientation as well as the pattern. **Data augmentation** enlarges the sample with transformed copies of its images — flips, rotations, shifts, small changes of scale and intensity — under the condition that the transformation **preserves the label**.<sup class="c"><a href="#r12">[12]</a></sup> The condition is the whole method: a chest radiograph flipped left–right still shows pneumonia, but a mirrored heart is *situs inversus*, a diagnosis of its own; a dermoscopic image can be rotated by any angle, because skin has no "up"; a vertical bar rotated by 90° becomes a horizontal bar, a different class. Augmentation also implements an **invariance** the designer believes in — "the label does not depend on a one-pixel shift" — and Section 6 measures how much of that invariance a perceptron acquires from it.



In [ ]:
shift_img <- function(m, dx, dy){ H<-nrow(m); W<-ncol(m); o<-matrix(0,H,W)                 # integer shift, zero fill
  for (r in 1:H) for (c in 1:W){ sr<-r-dy; sc<-c-dx; if (sr>=1&&sr<=H&&sc>=1&&sc<=W) o[r,c]<-m[sr,sc] }; o }
rotate_nn <- function(m, deg){ H<-nrow(m); ctr<-(H+1)/2; a<--deg*pi/180; o<-matrix(0,H,H) # nearest-neighbour rotation
  for (r in 1:H) for (c in 1:H){ sr<-round(ctr+(r-ctr)*cos(a)-(c-ctr)*sin(a)); sc<-round(ctr+(r-ctr)*sin(a)+(c-ctr)*cos(a))
    if (sr>=1&&sr<=H&&sc>=1&&sc<=H) o[r,c]<-m[sr,sc] }; o }
augment <- function(m, kind) switch(kind,
  "flip L-R"=m[,ncol(m):1], "flip U-D"=m[nrow(m):1,], "rotate 90"=t(m)[,nrow(m):1],
  "rotate 15"=rotate_nn(m,15), "shift 1px"=shift_img(m,1,1), "brighter"=pmin(m*1.3,1))
kinds <- c("flip L-R","flip U-D","rotate 90","rotate 15","shift 1px","brighter")
set.seed(3); vb <- draw_glyph("vbar"); pt <- nod$X[which(nod$y==1)[2],,]
par(mfrow=c(2,7), mar=c(1,1,2,1))
for (base in list(list(vb,"vbar glyph"), list(pt,"nodule patch"))){
  image(t(base[[1]][nrow(base[[1]]):1,]),col=grey.colors(32,1,0),axes=FALSE,main="original",cex.main=0.8); box()
  for (k in kinds){ image(t(augment(base[[1]],k)[nrow(base[[1]]):1,]),col=grey.colors(32,1,0),axes=FALSE,main=k,cex.main=0.8); box() } }
cat("Label preserved?  vbar glyph: flips/90-rotation NO (a flipped or turned bar is another class); small rotation/shift/brightness YES\n")
cat("                  patch: all six YES (a nodule is a nodule in any orientation and exposure)\n")



<p class="caption"><b>Figure 5.</b> Six augmentations of a glyph (top) and of a nodule patch (bottom). For the patch every transformation is admissible: the nodule remains a nodule. For the vertical-bar glyph, the flips and the quarter-turn change the class (a quarter-turned vertical bar is a horizontal bar), while the small rotation, the shift and the brightness change do not. Augmentation therefore requires a decision by someone who knows the domain, which is one of the reasons why a bioengineer, and not only a computer scientist, belongs on an imaging-AI team. Augmentation is applied only to the training folds; augmenting before splitting would place transformed copies of a test image in the training set, the leakage of Supplement S1 in a new disguise.</p>

<div class="exercise"><h4>Exercise 1</h4><p>In the augmentation chunk, replace the <code>vbar</code> by a <code>cross</code> and by a <code>ring</code> and decide, for each transformation, whether the label is preserved. (The <code>cross</code> and the <code>ring</code> are symmetric under the flips and the quarter-turn, so for them more transformations are admissible than for the bar — the admissible set depends on the symmetry of the class.) Then add a 180° rotation (<code>rotate_nn(m, 180)</code>) and list the glyph classes for which it preserves the label.</p></div>

## 3. Hand-crafted features versus learned features

Until about 2012, image classification meant designing features: numbers computed from the image by a rule the engineer wrote, suggested by a radiologist's description of the lesion — "brighter", "round", "heterogeneous texture" — and fed to the classifiers of Units 3 and 4. This section builds a small extractor of that kind, traces its texture component by hand, and trains k-nearest neighbours, a support vector machine and a random forest on it: the **classical baseline** against which the perceptron of Section 4 is judged.

### 3.1 First-order features

First-order features depend only on the distribution of intensities, not on their arrangement: the mean, standard deviation, skewness, kurtosis and range of the pixel values, and the **entropy** of the histogram, \( -\sum_v p(v) \log_2 p(v) \), which measures how spread the grey levels are. A nodule raises the skewness (a bright tail) and the maximum of the patch slightly; the texture features capture how the brightness is arranged.

### 3.2 Texture: the grey-level co-occurrence matrix, by hand

Haralick's co-occurrence matrix (GLCM) is the classical texture descriptor, still used in radiomics.<sup class="c"><a href="#r13">[13]</a></sup> Quantise the image to \( L \) grey levels, choose a displacement (for instance "one pixel to the right"), and count how often a pixel of level \( i \) has a neighbour of level \( j \) at that displacement. The counts, normalised to sum to one, form the \( L \times L \) matrix \( P(i,j) \); scalar features are then computed from it. Three of Haralick's fourteen suffice for this unit:

<div class="eq">\[ \text{contrast} = \sum_{i,j} (i-j)^2 P(i,j), \qquad \text{homogeneity} = \sum_{i,j} \frac{P(i,j)}{1+|i-j|}, \qquad \text{energy} = \sum_{i,j} P(i,j)^2 \]
<span class="where">Contrast is large when neighbouring pixels differ (edges, noise); homogeneity is large when they are alike (smooth regions); energy is large when a few transitions dominate (regular texture). A fourth feature, the correlation between \( i \) and \( j \) under \( P \), measures linear structure.</span></div>

Take a 4 × 4 image quantised to \( L = 4 \) levels and the displacement "one pixel to the right":

\[ \mathbf{I} = \begin{pmatrix} 0 & 0 & 1 & 1 \\ 0 & 0 & 1 & 1 \\ 0 & 2 & 2 & 2 \\ 2 & 2 & 3 & 3 \end{pmatrix} \]

Each row has three horizontal pairs \( (I_{r,c}, I_{r,c+1}) \), twelve in all: row 1 gives (0,0), (0,1), (1,1); row 2 the same; row 3 gives (0,2), (2,2), (2,2); row 4 gives (2,2), (2,3), (3,3). Counting:

| pair \( (i,j) \) | (0,0) | (0,1) | (0,2) | (1,1) | (2,2) | (2,3) | (3,3) |
|---|---|---|---|---|---|---|---|
| count | 2 | 2 | 1 | 2 | 3 | 1 | 1 |
| \( P(i,j) \) | 2/12 | 2/12 | 1/12 | 2/12 | 3/12 | 1/12 | 1/12 |

Then contrast \( = [2 \cdot 1^2 + 1 \cdot 2^2 + 1 \cdot 1^2]/12 = 7/12 \approx 0.583 \) (only the off-diagonal pairs contribute); homogeneity \( = [8 \cdot 1 + 2 \cdot \tfrac{1}{2} + 1 \cdot \tfrac{1}{3} + 1 \cdot \tfrac{1}{2}]/12 \approx 0.819 \); energy \( = (4+4+1+4+9+1+1)/144 = 24/144 \approx 0.167 \). (Haralick's original definition symmetrises the matrix by adding its transpose; the unsymmetrised form used here gives the same contrast and homogeneity and is simpler to trace.) The chunk confirms the arithmetic.



In [ ]:
glcm <- function(q, levels, off=c(0,1)){               # normalised co-occurrence matrix for a displacement (dy,dx)
  dy <- off[1]; dx <- off[2]; H <- nrow(q); W <- ncol(q)
  a <- q[1:(H-dy), 1:(W-dx)]; b <- q[(1+dy):H, (1+dx):W]; P <- matrix(0, levels, levels)
  for (k in seq_along(a)) P[a[k]+1, b[k]+1] <- P[a[k]+1, b[k]+1] + 1
  P / sum(P)
}
glcm_feats <- function(img, levels=8, offs=list(c(0,1), c(1,0))){
  q <- pmin(floor(minmax(img)*levels), levels-1); idx <- 0:(levels-1)
  I <- matrix(idx, levels, levels); J <- t(I); out <- NULL
  for (off in offs){ P <- glcm(q, levels, off); mi <- sum(P*I); mj <- sum(P*J)
    si <- sqrt(sum(P*(I-mi)^2)); sj <- sqrt(sum(P*(J-mj)^2))
    out <- rbind(out, c(sum(P*(I-J)^2), sum(P/(1+abs(I-J))), sum(P^2), sum(P*(I-mi)*(J-mj))/(si*sj+1e-12))) }
  colMeans(out)
}
Ih <- matrix(c(0,0,1,1, 0,0,1,1, 0,2,2,2, 2,2,3,3), 4, 4, byrow=TRUE)
P <- glcm(Ih, 4, c(0,1)); i <- matrix(0:3,4,4); j <- t(i)
cat("Co-occurrence counts (rows i, cols j), displacement one pixel to the right:\n"); print(round(P*12))
cat(sprintf("contrast = %.3f   homogeneity = %.3f   energy = %.3f\n",
            sum(P*(i-j)^2), sum(P/(1+abs(i-j))), sum(P^2)))



The counts and the three features agree with the hand calculation: contrast 0.583, homogeneity 0.819, energy 0.167. The function `glcm_feats` quantises an image to eight levels and averages the four features over the horizontal and vertical displacements, so that a texture rotated by a quarter turn produces the same numbers.

### 3.3 Shape features from a thresholded mask

Shape requires a **segmentation**: a binary mask that says which pixels belong to the object. The simplest segmentation is a threshold — here, pixels brighter than the patch mean plus 1.5 standard deviations after light smoothing — followed by **connected-component labelling**, which groups adjacent foreground pixels into objects. The chunk implements the labelling as a short flood-fill in base R (the R stand-in for `scipy.ndimage.label`). From the largest component we compute its area, its contrast against the background, and its eccentricity (from the eigenvalues of the covariance of its pixel coordinates: 0 for a disc, close to 1 for a line); the number of components is a fourth feature that distinguishes one nodule from a scatter of microcalcifications. This is a miniature of what the Wisconsin program did to cell nuclei in 1993.<sup class="c"><a href="#r9">[9]</a></sup> The chunk assembles the fourteen features and applies the extractor to the thousand nodule patches.



In [ ]:
label_cc <- function(mask){                              # 4-connectivity connected components (flood fill)
  H <- nrow(mask); W <- ncol(mask); lab <- matrix(0, H, W); cur <- 0
  for (r in 1:H) for (c in 1:W) if (mask[r,c] && lab[r,c]==0){ cur <- cur+1; st <- list(c(r,c))
    while (length(st)>0){ p <- st[[length(st)]]; st[[length(st)]] <- NULL; pr <- p[1]; pc <- p[2]
      if (pr<1||pr>H||pc<1||pc>W) next; if (!mask[pr,pc]||lab[pr,pc]!=0) next; lab[pr,pc] <- cur
      st <- c(st, list(c(pr-1,pc),c(pr+1,pc),c(pr,pc-1),c(pr,pc+1))) } }
  list(lab=lab, n=cur)
}
shape_feats <- function(img){ sm <- gauss_filter(img, 1.0); mask <- sm > mean(sm) + 1.5*sd(sm)
  cc <- label_cc(mask); if (cc$n==0) return(c(0,0,0,0))
  sizes <- tabulate(cc$lab[cc$lab>0], cc$n); k <- which.max(sizes); comp <- cc$lab==k
  ij <- which(comp, arr.ind=TRUE)
  ev <- if (nrow(ij)>2) sort(eigen(cov(ij), only.values=TRUE)$values, decreasing=TRUE) else c(1,1)
  c(sizes[k], cc$n, sqrt(max(1 - ev[2]/(ev[1]+1e-9), 0)), mean(sm[comp]) - mean(sm[!mask])) }
FEATURE_NAMES <- c("mean","sd","skewness","kurtosis","range","entropy",
                   "glcm_contrast","glcm_homogeneity","glcm_energy","glcm_correlation",
                   "area","n_objects","eccentricity","obj_contrast")
extract_features <- function(img){ v <- as.vector(img)
  h <- tabulate(pmin(floor(v*32)+1, 32), 32) / length(v); h <- h[h>0]
  first <- c(mean(v), sd(v), e1071::skewness(v), e1071::kurtosis(v), max(v)-min(v), -sum(h*log2(h)))
  c(first, glcm_feats(img), shape_feats(img)) }
nod_F <- t(sapply(1:dim(nod$X)[1], function(k) extract_features(nod$X[k,,])))
colnames(nod_F) <- FEATURE_NAMES
cat("feature matrix:", nrow(nod_F), "x", ncol(nod_F), "\n")
print(round(rbind(`no lesion`=colMeans(nod_F[nod$y==0,]), nodule=colMeans(nod_F[nod$y==1,])), 3))



The table of class means already tells a story. The first-order mean barely moves between the classes, as Figure 2 anticipated, while the shape and distribution features separate them: positive patches have a larger bright object (`area`) with higher contrast against its surroundings (`obj_contrast`), a positive `skewness` and higher `kurtosis` from the bright blob, and in the GLCM a lower contrast and higher homogeneity, because a smooth blob replaces part of the mottled background. These are the numbers a radiologist's description ("a well-circumscribed density") would have led an engineer to compute.

### 3.4 The classical baseline

The chunk trains the three classifiers of Units 3–4 — 5-nearest neighbours, an RBF support vector machine<sup class="c"><a href="#r37">[37]</a></sup> and a random forest of 200 trees<sup class="c"><a href="#r38">[38]</a></sup> — on the fourteen features, and, for comparison, on the 256 raw pixels, with stratified 5-fold cross-validation. The folds are fixed by the seed and are the same for every classifier, so that differences between rows are differences between methods and not between splits; Section 5 makes this paired design explicit. The helper `sc_fit` standardises on the training rows and applies the same centre and scale to the test rows, the in-fold scaling the warning box of Section 2.1 required.



In [ ]:
flat_patch <- function(Xarr) t(apply(Xarr, 1, function(m) as.vector(t(m))))   # flatten each patch, row by row
nod_P <- flat_patch(nod$X)                                                    # 1000 x 256
strat_folds <- function(y, k=5){ f <- integer(length(y))
  for (cl in unique(y)){ idx <- which(y==cl); f[idx] <- sample(rep(1:k, length.out=length(idx))) }; f }
sc_fit <- function(Xtr,ytr,Xte,model){ ce <- colMeans(Xtr); s <- apply(Xtr,2,sd); s[s==0] <- 1
  Ztr <- scale(Xtr,ce,s); Zte <- scale(Xte,ce,s)
  if (model=="knn") as.character(knn(Ztr,Zte,factor(ytr),k=5))
  else if (model=="svm") as.character(predict(svm(Ztr,factor(ytr)), Zte))
  else as.character(predict(randomForest(Ztr,factor(ytr),ntree=200), Zte)) }
cv_acc <- function(X,y,model,folds){ sapply(1:max(folds), function(f){
  te <- folds==f; mean(sc_fit(X[!te,],y[!te],X[te,],model)==as.character(y[te])) }) }
set.seed(0); folds <- strat_folds(nod$y, 5)
baseline <- sapply(c(`k-NN (k=5)`="knn", `SVM (RBF)`="svm", `random forest (200)`="rf"), function(m)
  c(`14 features`=sprintf("%.3f +/- %.3f", mean(cv_acc(nod_F,nod$y,m,folds)), sd(cv_acc(nod_F,nod$y,m,folds))),
    `256 pixels`=sprintf("%.3f +/- %.3f", mean(cv_acc(nod_P,nod$y,m,folds)), sd(cv_acc(nod_P,nod$y,m,folds)))))
cat("Nodule task, accuracy (mean +/- sd over 5 stratified folds):\n"); print(t(baseline), quote=FALSE)



On the fourteen designed features the three classifiers reach the high eighties to low nineties, the random forest the best of them; on the raw pixels every one does worse — the forest by a few points, the support vector machine by about ten, and k-nearest neighbours by more than twenty, because the Euclidean distance between 256-pixel vectors is dominated by background texture rather than by the nodule. The lesson is the one the field lived with for two decades: with a thousand images, *good features beat raw pixels*, and the quality of the system was the quality of the engineer's features. Keep the first column in mind — it is the bar the perceptron must clear.

<div class="box bridge"><h4>Bridge to radiomics and to BGM324</h4><p>The extractor above is a toy version of <strong>radiomics</strong>, the discipline that computes hundreds of standardised first-order, texture and shape features from segmented lesions on CT, MRI and PET and feeds them to the classifiers of Units 3–4 to predict histology, mutation status or survival. Its feature definitions are the direct descendants of Haralick's 1973 paper.<sup class="c"><a href="#r13">[13]</a></sup> The workflow is identical to the QSAR modelling of the bioinformatics course: an object (a lesion, a molecule) is described by a vector of descriptors, and the model is only as good — and as reproducible — as the descriptors are.</p></div>

## 4. The multilayer perceptron as an image classifier (8.1)

### 4.1 Flatten, standardise, train

Unit 6 built the multilayer perceptron: an input layer of \( p \) units, a hidden layer that computes \( h = \phi(\mathbf{W}\mathbf{x} + \mathbf{b}) \) with a non-linear activation \( \phi \), and a softmax output layer, fitted by minimising the cross-entropy loss. Nothing in that description mentions images: to classify an image one flattens it (Section 1.2), standardises each pixel position across the training set, and treats the result as a feature vector of length \( p = HW \). For the 8 × 8 glyphs \( p = 64 \); with a single hidden layer of \( m \) units the network has \( 64m + m \) weights into the hidden layer and \( 6m + 6 \) into the six-way output. R's standard perceptron is `nnet`,<sup class="c"><a href="#r35">[35]</a></sup> which fits a **single hidden layer** with a logistic activation by the BFGS optimiser; Section 5 uses `neuralnet` and a from-scratch trainer where more than one layer or a different optimiser is needed. The chunk evaluates the perceptron with the methodology of Supplement S1 — **repeated stratified 5-fold cross-validation**, two repetitions, giving ten estimates of the test accuracy from which a mean and a standard deviation are reported.



In [ ]:
make_glyphs <- function(per){ n <- per*length(GLYPHS); X <- matrix(0,n,SG*SG); y <- character(n); k <- 1
  for (cls in GLYPHS) for (i in 1:per){ X[k,] <- as.vector(draw_glyph(cls)); y[k] <- cls; k <- k+1 }
  list(X=X, y=factor(y, levels=GLYPHS)) }
set.seed(1); G <- make_glyphs(120)                      # 720 glyphs, 120 per class
Gctr <- colMeans(G$X); Gsd <- apply(G$X,2,sd); Gsd[Gsd==0] <- 1
Gs <- scale(G$X, Gctr, Gsd)                             # standardised once for display models
mlp_cv <- function(X, y, size, folds, decay=1e-3, maxit=200){      # out-of-fold predictions, multiclass nnet
  pred <- factor(rep(NA, length(y)), levels=levels(y))
  for (f in 1:max(folds)){ te <- folds==f; ce <- colMeans(X[!te,]); s <- apply(X[!te,],2,sd); s[s==0] <- 1
    fit <- nnet(scale(X[!te,],ce,s), class.ind(y[!te]), size=size, softmax=TRUE, maxit=maxit,
                trace=FALSE, decay=decay, MaxNWts=1e5)
    pred[te] <- factor(predict(fit, scale(X[te,],ce,s), type="class"), levels=levels(y)) }
  pred }
set.seed(10)
scores <- unlist(lapply(1:2, function(rep){ fo <- strat_folds(G$y,5); oof <- mlp_cv(G$X,G$y,16,fo)
  sapply(1:5, function(f) mean(oof[fo==f]==G$y[fo==f])) }))
set.seed(0); fit_all <- nnet(Gs, class.ind(G$y), size=16, softmax=TRUE, maxit=200, trace=FALSE, decay=1e-3, MaxNWts=1e5)
npar <- length(fit_all$wts)
cat(sprintf("architecture 64 -> 16 -> 6 : %d trainable parameters for %d images\n", npar, length(G$y)))
cat("accuracy per split:", round(scores,3), "\n")
cat(sprintf("repeated stratified 5-fold CV accuracy: %.3f +/- %.3f  (min %.3f, max %.3f)\n",
            mean(scores), sd(scores), min(scores), max(scores)))



The perceptron classifies the great majority of held-out glyphs correctly, with a standard deviation across the ten splits of a couple of percentage points. Note what the standard deviation is *for*: a network that scored a point higher on a single split would not be demonstrably better than this one, because single-split differences of that size occur between repetitions of the same model. This is why Section 5 reports every comparison with its spread across folds.

### 4.2 Which classes are confused

Accuracy summarises; the confusion matrix (Supplement S1) explains. The chunk obtains an out-of-fold prediction for every glyph — each predicted by a model that did not train on it — and tabulates true against predicted class.



In [ ]:
set.seed(2); foldsG <- strat_folds(G$y, 5)
y_oof <- mlp_cv(G$X, G$y, 16, foldsG)
C <- table(truth=G$y, predicted=y_oof)
cat(sprintf("out-of-fold accuracy: %.3f | errors: %d of %d\n",
            mean(y_oof==G$y), sum(y_oof!=G$y), length(G$y)))
off <- which(C>0 & row(C)!=col(C), arr.ind=TRUE)
ord <- order(-C[off]); top <- head(off[ord,,drop=FALSE], 5)
cat("most frequent confusions (true -> predicted : count): ",
    paste(sprintf("%s->%s: %d", GLYPHS[top[,1]], GLYPHS[top[,2]], C[top]), collapse=", "), "\n")
recall <- round(diag(C)/rowSums(C), 3); cat("per-class recall:", paste(GLYPHS, recall, sep="="), "\n")
par(mar=c(5,5,3,1))
image(1:6, 1:6, t(C[6:1,]), col=colorRampPalette(c("white", palette_neg))(32), axes=FALSE,
      xlab="predicted", ylab="true", main="Confusion matrix (out-of-fold, 5-fold CV)")
axis(1, 1:6, GLYPHS, las=2); axis(2, 1:6, rev(GLYPHS), las=1)
for (a in 1:6) for (b in 1:6) text(b, 7-a, C[a,b], cex=0.9, col=if (C[a,b]>max(C)/2) "white" else "black")



<p class="caption"><b>Figure 6.</b> The confusion matrix of the perceptron on the 720 glyphs. The errors are not spread evenly: the <code>ring</code> is the most confusable class, because at 8 × 8 pixels with heavy noise a noisy ring resembles a <code>cross</code> or a bar, and the <code>cross</code> shares its vertical and horizontal strokes with <code>vbar</code> and <code>hbar</code>. The per-class recall printed above the figure ranges from the high eighties for the <code>ring</code> to near one for the cleanly separated bars. A radiology analogue would be a confusion matrix of pneumonia, oedema and atelectasis, three causes of opacity that radiologists themselves confuse; the matrix shows where a second reader is needed.</p>



In [ ]:
wrong <- which(y_oof != G$y)[1:min(20, sum(y_oof!=G$y))]
par(mfrow=c(2,10), mar=c(0.3,0.3,1.8,0.3), oma=c(0,0,2,0))
for (k in wrong){ image(t(matrix(G$X[k,],SG,SG)[SG:1,]), col=grey.colors(32,1,0), axes=FALSE, zlim=c(0,1))
  title(sprintf("%s -> %s", substr(as.character(G$y[k]),1,4), substr(as.character(y_oof[k]),1,4)), cex.main=0.75); box() }
mtext(sprintf("The first %d of the %d out-of-fold errors (true -> said)", length(wrong), sum(y_oof!=G$y)),
      outer=TRUE, line=0.3, cex=0.9, font=2)



<p class="caption"><b>Figure 7.</b> Twenty of the misclassified glyphs with the true label and the network's answer. Several are ambiguous even to a human at this resolution and noise level — a faint ring broken by noise is read as a cross, a short bar as a diagonal — which is the point: at eight pixels per side the information is sometimes not in the image, and no classifier can recover it. Looking at the errors is the cheapest diagnostic in the field, and it is the step at which the hospital markers of Section 7 were discovered in real datasets.</p>

### 4.3 What the first layer has learned: weights as templates

Each hidden unit of the first layer computes \( \phi(\mathbf{w}_k^\top \mathbf{x} + b_k) \), a weighted sum over the 64 pixels. The weight vector \( \mathbf{w}_k \) has one entry per pixel and can therefore be reshaped into an 8 × 8 image and displayed. A unit whose weights are positive where a stroke is expected and negative elsewhere acts as a **template** — a matched filter for a shape configuration — exactly like the class-mean templates of Unit 1, except that the templates are now learned to be *useful for discrimination* rather than to be averages, and 16 of them are combined by the output layer.



In [ ]:
# extract the 64 input weights of each hidden unit from the nnet weight vector (layout: per hidden unit, bias then inputs)
p <- SG*SG; m <- 16
W1 <- sapply(1:m, function(k) fit_all$wts[((k-1)*(p+1)+2):(k*(p+1))])   # 64 x 24
ord <- order(-colSums(abs(W1)))
vmax <- max(abs(W1))
par(mfrow=c(2,8), mar=c(0.3,0.3,1.4,0.3), oma=c(0,0,2,0))
rb <- colorRampPalette(c(palette_neg,"white",palette_pos))(32)
for (k in ord){ image(t(matrix(W1[,k],SG,SG)[SG:1,]), col=rb, axes=FALSE, zlim=c(-vmax,vmax))
  title(sprintf("unit %d", k), cex.main=0.75); box() }
mtext("First-layer weights as 8 x 8 templates (blue negative, orange positive); all 16 units",
      outer=TRUE, line=0.3, cex=0.9, font=2)
cat(sprintf("weight matrix: %d x %d | weights range [%.2f, %.2f]\n", nrow(W1), ncol(W1), min(W1), max(W1)))



<p class="caption"><b>Figure 8.</b> The 16 first-layer units, each shown as the 8 × 8 image of its weights. Some are recognisable stroke detectors — a positive (orange) vertical ridge with negative (blue) flanks responds to a vertical bar; a diagonal band responds to <code>diagL</code> or <code>diagR</code> — and many are noisy combinations useful only in concert with the others. The templates are <em>global</em>: each unit looks at all 64 pixels with its own fixed weights, so a stroke detector that fires for a bar on the left is a different unit from one that detects the same bar on the right. Section 6 shows that this is exactly what makes the perceptron fragile to shifts and why convolutional networks share weights across positions.</p>

### 4.4 The perceptron on the simulated nodules

The chunk repeats the comparison of Section 3.4 with the perceptron added: once on the 256 raw pixels, once on the 14 designed features. For the pixel model a strong L2 penalty (`decay = 0.5`) is used, because with 256 inputs and 1,000 images the network overfits severely at a small penalty; this choice is itself a topology decision of the kind Section 5 studies systematically. `nnet` is trained for a binary outcome with `entropy = TRUE` (logistic loss).



In [ ]:
mlp_bin_cv <- function(X, y, size, folds, decay){ p <- numeric(length(y))
  for (f in 1:max(folds)){ te <- folds==f; ce <- colMeans(X[!te,]); s <- apply(X[!te,],2,sd); s[s==0] <- 1
    fit <- nnet(scale(X[!te,],ce,s), y[!te], size=size, entropy=TRUE, maxit=250, trace=FALSE, decay=decay, MaxNWts=1e5)
    p[te] <- as.vector(predict(fit, scale(X[te,],ce,s))) }; p }
set.seed(0); pF <- mlp_bin_cv(nod_F,nod$y,6,folds,0.5); pP <- mlp_bin_cv(nod_P,nod$y,6,folds,0.5)
accF <- sapply(1:5, function(f) mean((pF[folds==f]>0.5)==nod$y[folds==f]))
accP <- sapply(1:5, function(f) mean((pP[folds==f]>0.5)==nod$y[folds==f]))
mlp_rows <- rbind(`MLP (6), decay=0.5` = c(`14 features`=sprintf("%.3f +/- %.3f",mean(accF),sd(accF)),
                                           `256 pixels`=sprintf("%.3f +/- %.3f",mean(accP),sd(accP))))
cat("Nodule task, accuracy (mean +/- sd over the same 5 stratified folds as Section 3.4):\n")
print(rbind(t(baseline), mlp_rows), quote=FALSE)



On the designed features the perceptron sits among the classical classifiers — the features carry the information, and any reasonable classifier finds a boundary. On raw pixels the perceptron with the strong penalty reaches the low-to-mid eighties, far better than k-nearest neighbours on pixels and comparable to the support vector machine, but at or below the random forest on pixels and **below the feature-based models**. This is the state of the art of about 2010 in miniature: a shallow network on pixels does not beat good features with a thousand training images. The lead of the designed features is a lead of *prior knowledge* — someone knew that a nodule is a bright, round, locally contrasting object and wrote that down as code. Section 6 shows how to give a network comparable prior knowledge about images without hand-designing the features.

### 4.5 Probabilities and thresholds in a screening setting

A perceptron outputs a probability \( \hat{p}(\text{nodule} \mid \mathbf{x}) \); the label is assigned when \( \hat{p} \) exceeds a threshold. The default of 0.5 is appropriate when both errors cost the same and the training prevalence matches deployment; neither holds in screening, where a missed cancer costs more than a recall and prevalence is a few per cent. The chunk computes sensitivity, specificity and the ROC curve<sup class="c"><a href="#r39">[39]</a></sup> from out-of-fold probabilities and converts them to positive predictive values at two prevalences with Bayes' theorem (Unit 3): \( \mathrm{PPV} = \frac{\text{se}\,\pi}{\text{se}\,\pi + (1-\text{sp})(1-\pi)} \).



In [ ]:
p_oof <- pP                                 # reuse the pixel-MLP out-of-fold probabilities from Section 4.4
roc_obj <- roc(nod$y, p_oof, quiet=TRUE, direction="<")
ppv <- function(se,sp,pi) se*pi/(se*pi + (1-sp)*(1-pi))
ops <- do.call(rbind, lapply(c(0.1,0.3,0.5,0.7,0.9), function(t){ pred <- p_oof>=t
  se <- sum(pred & nod$y==1)/sum(nod$y==1); sp <- sum(!pred & nod$y==0)/sum(nod$y==0)
  data.frame(threshold=t, sensitivity=round(se,3), specificity=round(sp,3),
             `PPV_prev50`=round(ppv(se,sp,0.5),3), `PPV_prev2`=round(ppv(se,sp,0.02),3)) }))
cat(sprintf("ROC-AUC of the out-of-fold probabilities: %.3f\n", as.numeric(auc(roc_obj))))
plot(1-ops$specificity, ops$sensitivity, type="n", xlim=c(0,1), ylim=c(0,1),
     xlab="1 - specificity", ylab="sensitivity", main="ROC curve of the pixel perceptron")
lines(1-roc_obj$specificities, roc_obj$sensitivities, col=palette_neg, lwd=2)
abline(0,1,lty=2, col="grey"); points(1-ops$specificity, ops$sensitivity, pch=19, col=palette_pos)
text(1-ops$specificity, ops$sensitivity, paste0("t=",ops$threshold), pos=4, cex=0.7)
legend("bottomright", sprintf("pixel MLP, AUC = %.3f", as.numeric(auc(roc_obj))), lwd=2, col=palette_neg, bty="n", cex=0.8)
print(ops, row.names=FALSE)



<p class="caption"><b>Figure 9.</b> The ROC curve of the pixel perceptron on the nodule task, built from out-of-fold probabilities, with five thresholds marked. Lowering the threshold from 0.5 to 0.1 moves the operating point up and to the right: sensitivity rises and specificity falls, as the table shows. The last two columns are the consequence of prevalence that Unit 3 derived: at the 50 % prevalence of this balanced dataset the positive predictive value is high at every threshold, but at a screening prevalence of 2 % most positive calls are false at any threshold, because the false-positive rate is applied to the large healthy majority. The curve and its AUC are properties of the model; the PPV is a property of the model <em>and</em> the population, and a screening tool must be evaluated at the prevalence where it will be used.</p>

<div class="exercise"><h4>Exercise 2</h4><p>Choose the threshold that gives a sensitivity of at least 0.95 (read it from <code>roc_obj</code> with <code>coords(roc_obj, "all")</code>) and compute the number of false positives per 1,000 screened patients at 2 % prevalence. Then repeat with the feature-based random forest of Section 3.4 (use its class-probability output, <code>predict(..., type="prob")</code>). Which model would you deploy as a <em>second reader</em> whose positives are reviewed by a radiologist, and which as a <em>triage</em> tool that discards negatives unread?</p></div>

## 5. Experimental tests of network topology (8.2)

### 5.1 How to run an honest experiment

The topology of a perceptron — hidden layers, units per layer, activation, regularisation strength, optimiser, learning rate, stopping rule — is not learned from the data; it is chosen by the engineer, and the course plan asks you to choose it *experimentally*. Four rules, all from Supplement S1, make such an experiment trustworthy:

1. **Same splits for every configuration** (fixed by `set.seed`), so that differences between configurations are not confounded with differences between splits: the comparison is *paired*.
2. **Repeat, and report the variance.** A difference smaller than the standard deviations across splits is not a finding.
3. **Change one factor at a time**, or cross the factors in a factorial design, as in the depth × width grid below.
4. **Do not report the best cross-validated score as the expected performance.** Choosing the best of many configurations on the same folds fits those folds, and the chosen score is optimistically biased;<sup class="c"><a href="#r15">[15]</a></sup> Section 5.6 shows the remedy.

<div class="box warn"><h4>One R detail that shapes this section</h4><p>R's <code>nnet</code> fits <em>one</em> hidden layer with a logistic activation, optimised by BFGS (a full-batch quasi-Newton method). It has no notion of depth beyond one layer, of a rectified-linear activation, or of a learning rate and mini-batches. To vary those factors faithfully this section uses three tools: <code>nnet</code> for the <strong>width</strong> experiment (Section 5.2) and the <strong>regularisation</strong> experiment (its <code>decay</code> argument is the L2 penalty); <code>neuralnet</code><sup class="c"><a href="#r36">[36]</a></sup> for the <strong>depth</strong> experiment, which supports several hidden layers and a choice of logistic or <code>tanh</code> activation; and a <strong>from-scratch</strong> mini-batch gradient-descent trainer (Section 5.5) for the learning-rate, momentum and batch-size experiment, which also lets us use the rectified linear unit that <code>nnet</code> lacks. The Python edition used a single <code>MLPClassifier</code> for all of these; the R edition splits them across the three tools whose union covers the same ground.</p></div>

### 5.2 Depth × width

The first experiment crosses the number of hidden layers (0, 1, 2, 3) with the number of units per layer (8, 16, 24). A network with zero hidden layers is a multinomial logistic regression — the linear baseline. To keep the multi-layer optimisation fast and reliably convergent, the depth grid trains `neuralnet` on a 12-component PCA of the glyph pixels (an unsupervised representation fixed in advance, so it introduces no label leakage); the single-layer `nnet` on raw pixels of Section 4.1 reached a comparable accuracy, which ties the two representations together. The chunk fits the grid with 3-fold cross-validation and draws it as a heat-map.



In [ ]:
pc <- prcomp(G$X, center=TRUE, scale.=FALSE); Z <- scale(pc$x[,1:12]); colnames(Z) <- paste0("z",1:12)
Yoh <- class.ind(G$y); colnames(Yoh) <- GLYPHS
dz <- data.frame(Z, Yoh, check.names=FALSE)
form <- as.formula(paste(paste(GLYPHS, collapse="+"), "~", paste0("z",1:12, collapse="+")))
nn_cv <- function(hidden, folds){ sapply(1:max(folds), function(f){ te <- folds==f
  if (is.null(hidden)){ m <- multinom(cls ~ ., data=data.frame(Z, cls=G$y)[!te,], trace=FALSE)
    pr <- predict(m, data.frame(Z)[te,]) }
  else { nn <- try(neuralnet(form, dz[!te,], hidden=hidden, linear.output=FALSE, act.fct="logistic",
                             stepmax=1e5, threshold=0.1), silent=TRUE)
    if (inherits(nn,"try-error") || is.null(nn$weights)) return(NA)
    pr <- factor(GLYPHS[max.col(predict(nn, dz[te,]))], levels=GLYPHS) }
  mean(pr==G$y[te]) }) }
set.seed(3); fo <- strat_folds(G$y, 3); widths <- c(8,16,24)
M <- matrix(NA, 4, 3, dimnames=list(c("0 (linear)","1 layer","2 layers","3 layers"), paste0(widths," u")))
Sg <- M
lin <- nn_cv(NULL, fo); M[1,] <- mean(lin); Sg[1,] <- sd(lin)
for (j in seq_along(widths)){ w <- widths[j]
  for (d in 1:3){ a <- nn_cv(rep(w, d), fo); M[d+1,j] <- mean(a, na.rm=TRUE); Sg[d+1,j] <- sd(a, na.rm=TRUE) } }
best <- which(M==max(M,na.rm=TRUE), arr.ind=TRUE)[1,]
cat(sprintf("best cell: %s, %s -> %.3f | linear baseline %.3f\n",
            rownames(M)[best[1]], colnames(M)[best[2]], max(M,na.rm=TRUE), M[1,1]))
print(round(M,3))
par(mar=c(4,7,2,1)); image(1:3, 1:4, t(M[4:1,]), col=colorRampPalette(c("white",palette_acc))(32),
      axes=FALSE, xlab="units per layer", ylab=""); box()
axis(1, 1:3, paste0(widths)); axis(2, 1:4, rev(rownames(M)), las=1)
for (a in 1:4) for (b in 1:3) text(b, 5-a, sprintf("%.2f\n+/-%.2f", M[a,b], Sg[a,b]), cex=0.75)
title("Glyphs: CV accuracy by depth and width", cex.main=0.95)



<p class="caption"><b>Figure 10.</b> The depth × width grid on the glyphs (the linear model has no width and is repeated across its row). Three observations. First, the large and reliable jump is from the linear model to a single hidden layer: a hidden layer lifts the accuracy well above the linear baseline, because the classes are not linearly separable in the PC space. Second, adding further layers produces only a small additional rise here — the widest three-layer network is the best cell — but the increments from one layer to two or three are comparable to the fold-to-fold standard deviations printed in each cell, so the grid does not robustly rank the multi-layer configurations: on this sample they are, within noise, equivalent. Third, that is exactly the expected behaviour with a few hundred images — extra depth adds capacity the data can barely constrain, so any gain is small and uncertain. "Bigger is better" is a slogan about networks trained on millions of images; with a small sample the honest report is the whole grid with its error bars, not the single best cell, whose value is biased upward precisely because it was the maximum (Section 5.6).</p>

### 5.3 Learning curves: how performance depends on the sample

A learning curve plots cross-validated accuracy against the number of training images. It answers the question the grid cannot: *would more data help, and which model would benefit?* The chunk draws the curves for a narrow and a wide single-layer `nnet`, with the training accuracy alongside.



In [ ]:
lc <- function(size, fracs=c(0.15,0.35,0.6,1.0)){ set.seed(5); fo <- strat_folds(G$y,3)
  t(sapply(fracs, function(fr){ tr_acc <- cv_acc <- numeric(3)
    for (f in 1:3){ te <- fo==f; tri <- sample(which(!te)); tri <- tri[1:round(fr*length(tri))]
      ce <- colMeans(G$X[tri,]); s <- apply(G$X[tri,],2,sd); s[s==0] <- 1
      fit <- nnet(scale(G$X[tri,],ce,s), class.ind(G$y[tri]), size=size, softmax=TRUE, maxit=200,
                  trace=FALSE, decay=1e-3, MaxNWts=1e5)
      tr_acc[f] <- mean(predict(fit, scale(G$X[tri,],ce,s), type="class")==G$y[tri])
      cv_acc[f] <- mean(predict(fit, scale(G$X[te,],ce,s), type="class")==G$y[te]) }
    c(n=round(fr*length(G$y)*2/3), train=mean(tr_acc), cv=mean(cv_acc), sd=sd(cv_acc)) })) }
L8 <- lc(6); L128 <- lc(32)
plot(L8[,"n"], L8[,"cv"], type="b", pch=19, col=palette_pos, ylim=c(0.4,1.02),
     xlab="number of training images", ylab="accuracy", main="Learning curves on the glyphs")
lines(L8[,"n"], L8[,"train"], lty=2, col=palette_pos); lines(L128[,"n"], L128[,"cv"], type="b", pch=19, col=palette_neg)
lines(L128[,"n"], L128[,"train"], lty=2, col=palette_neg)
legend("bottomright", c("6 units, CV","6 units, train","32 units, CV","32 units, train"),
       col=c(palette_pos,palette_pos,palette_neg,palette_neg), lty=c(1,2,1,2), pch=c(19,NA,19,NA), bty="n", cex=0.75)
cat("CV accuracy by training size:\n"); print(round(rbind(`6 units`=L8[,"cv"], `32 units`=L128[,"cv"]),3))



<p class="caption"><b>Figure 11.</b> Learning curves of the 6-unit (orange) and 32-unit (blue) perceptrons. The wide network is ahead at every sample size by a few points, and it is still improving at the full training set, while its training accuracy (dashed) stays high — so the gap between the dashed and solid blue lines, the overfitting, closes only as the sample grows. The narrow network rises more slowly and plateaus several points lower, because it is limited by its own capacity (underfitting), and more data would help it less. The curves are the standard diagnostic for deciding whether to collect more images, which in medical imaging costs months and ethics approvals, or to change the model.</p>

### 5.4 Activation and regularisation

Two further factors. The **L2 penalty** (`decay` in `nnet`) shrinks the weights towards zero, the perceptron's equivalent of the ridge penalty of Unit 2; it is varied over five orders of magnitude on the 32-unit network. The **activation** is the logistic sigmoid or the hyperbolic tangent; `neuralnet` offers both, and the chunk compares them on the PC representation. (The rectified linear unit, which neither `nnet` nor `neuralnet` provides as a differentiable built-in, is used in the from-scratch trainer of Section 5.5, where its effect on training speed is visible.)



In [ ]:
set.seed(6); fo <- strat_folds(G$y, 3); decays <- c(1e-3, 1e-1, 1, 10, 100)
vc <- sapply(decays, function(al){ tr <- cv <- numeric(3)
  for (f in 1:3){ te <- fo==f; ce <- colMeans(G$X[!te,]); s <- apply(G$X[!te,],2,sd); s[s==0] <- 1
    fit <- nnet(scale(G$X[!te,],ce,s), class.ind(G$y[!te]), size=24, softmax=TRUE, maxit=200,
                trace=FALSE, decay=al, MaxNWts=1e5)
    tr[f] <- mean(predict(fit, scale(G$X[!te,],ce,s), type="class")==G$y[!te])
    cv[f] <- mean(predict(fit, scale(G$X[te,],ce,s), type="class")==G$y[te]) }
  c(train=mean(tr), cv=mean(cv), sd=sd(cv)) })
act <- sapply(c("logistic","tanh"), function(a){ acc <- numeric(3)
  for (f in 1:3){ te <- fo==f
    nn <- try(neuralnet(form, dz[!te,], hidden=16, linear.output=FALSE, act.fct=a, stepmax=1e5, threshold=0.1), silent=TRUE)
    acc[f] <- if (inherits(nn,"try-error")||is.null(nn$weights)) NA else
      mean(factor(GLYPHS[max.col(predict(nn, dz[te,]))], levels=GLYPHS)==G$y[te]) }
  mean(acc, na.rm=TRUE) })
plot(decays, vc["cv",], log="x", type="b", pch=19, col=palette_neg, ylim=c(min(vc)-0.02,1.0),
     xlab="decay (L2 penalty)", ylab="accuracy", main="Validation curve over the L2 penalty")
lines(decays, vc["train",], type="b", pch=0, lty=2, col="grey40")
arrows(decays, vc["cv",]-vc["sd",], decays, vc["cv",]+vc["sd",], angle=90, code=3, length=0.03, col=palette_neg)
legend("bottomleft", c("CV accuracy +/- sd","training accuracy"), col=c(palette_neg,"grey40"), lty=c(1,2), pch=c(19,0), bty="n", cex=0.8)
cat("CV accuracy by decay:", paste(decays, round(vc["cv",],3), sep="="), "\n")
cat(sprintf("activation (neuralnet, 16 units): logistic %.3f | tanh %.3f\n", act["logistic"], act["tanh"]))



<p class="caption"><b>Figure 12.</b> Training (grey) and cross-validated (blue) accuracy as the L2 penalty grows. The curve is roughly flat for small penalties — on these glyphs the network is not short of regularisation — then falls at the largest values, where the penalty shrinks the templates of Figure 8 towards zero and the training accuracy drops with them. The printed activation comparison shows the logistic and <code>tanh</code> units reaching similar accuracy here; on larger, noisier data the ordering can change, which is why these are experiments and not rules.</p>

### 5.5 Optimiser, learning rate and batch size

The weights are fitted by gradient descent (Units 2 and 6): a **mini-batch** of images is passed through the network, the gradient of the loss is computed by backpropagation, and the weights move against it by a step proportional to the **learning rate**; **momentum** adds a fraction of the previous step, which accelerates descent along consistent directions. Because `nnet` hides all of this inside BFGS, the chunk implements a compact one-hidden-layer softmax network with a rectified-linear activation **from scratch**, in a dozen lines of matrix code, and trains it under several settings on a single 70/30 split, plotting the training loss per epoch — the **loss curve**, the first thing to look at when training misbehaves.



In [ ]:
relu <- function(x) pmax(x,0)
softmax_rows <- function(Z){ Z <- Z - apply(Z,1,max); E <- exp(Z); E/rowSums(E) }
train_mlp <- function(X, Y, hidden=16, lr=0.1, batch=32, epochs=25, momentum=0){
  n <- nrow(X); p <- ncol(X); K <- ncol(Y)
  W1 <- matrix(rnorm(p*hidden,0,sqrt(2/p)),p,hidden); b1 <- rep(0,hidden)
  W2 <- matrix(rnorm(hidden*K,0,sqrt(2/hidden)),hidden,K); b2 <- rep(0,K)
  vW1<-0*W1; vb1<-0*b1; vW2<-0*W2; vb2<-0*b2; loss <- numeric(epochs)
  for (ep in 1:epochs){ idx <- sample(n); el <- 0
    for (s in seq(1,n,by=batch)){ bi <- idx[s:min(s+batch-1,n)]; xb <- X[bi,,drop=FALSE]; yb <- Y[bi,,drop=FALSE]; mb <- nrow(xb)
      H <- relu(sweep(xb%*%W1,2,b1,"+")); O <- softmax_rows(sweep(H%*%W2,2,b2,"+")); el <- el - sum(yb*log(O+1e-9))
      dO <- (O-yb)/mb; dW2 <- t(H)%*%dO; db2 <- colSums(dO); dH <- (dO%*%t(W2))*(H>0); dW1 <- t(xb)%*%dH; db1 <- colSums(dH)
      vW1<-momentum*vW1-lr*dW1; W1<-W1+vW1; vb1<-momentum*vb1-lr*db1; b1<-b1+vb1
      vW2<-momentum*vW2-lr*dW2; W2<-W2+vW2; vb2<-momentum*vb2-lr*db2; b2<-b2+vb2 }
    loss[ep] <- el/n }
  list(loss=loss, acc=function(Xn,yn) mean(GLYPHS[max.col(softmax_rows(sweep(relu(sweep(Xn%*%W1,2,b1,"+"))%*%W2,2,b2,"+")))]==yn)) }
set.seed(2); tr <- sample(nrow(Gs), round(0.7*nrow(Gs))); Yg <- class.ind(G$y)
par(mfrow=c(1,2), mar=c(4,4,2,1))
for (mom in c(0, 0.9)){ plot(NA, xlim=c(1,25), ylim=c(1e-3,3), log="y", xlab="epoch", ylab="training loss",
    main=sprintf("%s", ifelse(mom==0,"plain SGD","SGD + momentum 0.9"))); cols <- c(palette_pos,palette_neg,palette_acc); i <- 1
  for (lr in c(0.01,0.1,0.5)){ set.seed(1); m <- train_mlp(Gs[tr,], Yg[tr,], lr=lr, batch=32, epochs=25, momentum=mom)
    lines(m$loss, col=cols[i], lwd=1.3); legend_txt <- sprintf("lr %.2f: test acc %.2f", lr, m$acc(Gs[-tr,],G$y[-tr])) 
    text(25, m$loss[25], labels=legend_txt, pos=2, cex=0.6, col=cols[i]); i <- i+1 } }



<p class="caption"><b>Figure 13.</b> Training loss per epoch for plain mini-batch gradient descent (left) and the same with momentum 0.9 (right), at three learning rates, with the test accuracy after 25 epochs annotated on each curve. With plain descent the smallest learning rate has barely begun to converge; the middle rate is well-behaved; the largest converges fastest but less smoothly. Momentum accelerates the small learning rates dramatically — the 0.01 curve now reaches a low loss — but with the largest rate it overshoots and the loss diverges, the characteristic signature of a step that is too long. How a network is <em>trained</em> matters as much as how it is <em>shaped</em>, and a paper that reports an architecture without its optimiser settings has not described a reproducible experiment.</p>

### 5.6 Nested cross-validation: choosing the topology without flattering it

If the best cell of the grid of Section 5.2 is reported as "the accuracy of our perceptron", the number is biased upwards: it was selected *because* it was the highest among several noisy estimates on those same folds. Varma and Simon measured this bias on small biomedical datasets and found it can exceed the true differences between methods.<sup class="c"><a href="#r15">[15]</a></sup> The remedy is **nested cross-validation**: an *outer* loop holds out a test fold; inside each training fold an *inner* cross-validation chooses the topology (here, the width of a single hidden layer); the chosen model is refitted on the whole training fold and scored once on the outer test fold. The outer scores estimate the performance of the *procedure* "choose the width by cross-validation and train", which is what will actually be deployed.



In [ ]:
cand <- c(6, 16, 24)
inner_choose <- function(Xtr, ytr){ set.seed(1); ifo <- strat_folds(ytr, 3)
  acc <- sapply(cand, function(w) mean(sapply(1:3, function(f){ te <- ifo==f
    ce <- colMeans(Xtr[!te,]); s <- apply(Xtr[!te,],2,sd); s[s==0] <- 1
    fit <- nnet(scale(Xtr[!te,],ce,s), class.ind(ytr[!te]), size=w, softmax=TRUE, maxit=150, trace=FALSE, decay=1e-3, MaxNWts=1e5)
    mean(predict(fit, scale(Xtr[te,],ce,s), type="class")==ytr[te]) })))
  cand[which.max(acc)] }
set.seed(7); ofo <- strat_folds(G$y, 3); outer_acc <- numeric(3); chosen <- integer(3)
for (f in 1:3){ te <- ofo==f; w <- inner_choose(G$X[!te,], G$y[!te]); chosen[f] <- w
  ce <- colMeans(G$X[!te,]); s <- apply(G$X[!te,],2,sd); s[s==0] <- 1
  fit <- nnet(scale(G$X[!te,],ce,s), class.ind(G$y[!te]), size=w, softmax=TRUE, maxit=200, trace=FALSE, decay=1e-3, MaxNWts=1e5)
  outer_acc[f] <- mean(predict(fit, scale(G$X[te,],ce,s), type="class")==G$y[te]) }
cat(sprintf("nested CV accuracy of 'choose width by inner CV, then train': %.3f +/- %.3f\n", mean(outer_acc), sd(outer_acc)))
cat("width chosen in each outer fold:", chosen, "\n")
cat(sprintf("for comparison, the best single-layer cell of the grid (selected on the same data): %.3f\n", max(M[2,], na.rm=TRUE)))



The inner loop's chosen widths are printed for each outer fold, and the nested estimate is close to, and typically slightly below, the best single-layer cell of the grid. The gap is small here because the grid was small and the signal strong; on a dataset of a few hundred patients with a grid of fifty configurations, the gap is routinely several percentage points, and it is the difference between a published accuracy and the accuracy a hospital will observe. The principle is that of Supplement S1 taken one level up: whatever is *chosen* using data must be chosen inside the loop that *evaluates*.

<div class="box key"><h4>What the topology experiments established</h4>
<ul>
<li>On a few hundred small images the decisive gain is from the linear model to a single hidden layer; adding further layers changes the accuracy by no more than the fold-to-fold noise, so depth cannot be shown to help at this sample size.</li>
<li>Every comparison needs the same splits, repetition, and a standard deviation; most of the differences in this section are within one standard deviation of each other.</li>
<li>The optimiser, learning rate, momentum and penalty change the result by as much as the architecture does, and must be reported with it.</li>
<li>The configuration is chosen by inner cross-validation and the performance is estimated by outer cross-validation; the best cell of a grid is not an estimate of anything.</li>
</ul></div>

<div class="exercise"><h4>Exercise 3</h4><p>Add a width of 48 to the grid of Section 5.2 (one and two hidden layers only, to save time) and to the candidate set of the nested cross-validation. Does the wider network improve the nested estimate? Then reduce the dataset to 40 glyphs per class (<code>make_glyphs(40)</code>) and rerun the one-hidden-layer row of the grid. Which width wins now, and how wide are the standard deviations?</p></div>

## 6. Translation invariance, convolution, and why the perceptron is the wrong shape for images

### 6.1 Shift a glyph by one pixel

A perceptron's first-layer unit multiplies each pixel by its own weight (Figure 8). If the image is shifted by one pixel, every pixel now meets the weight intended for its neighbour, and the unit's response changes completely, although the picture is "the same" to any observer. The property a classifier of images should have — the class does not change when the object moves — is called **translation invariance**, and a perceptron has none of it except what it learns, position by position, from the training sample. The chunk measures the effect: a perceptron is trained on the glyphs in their usual positions and tested on a fresh set of glyphs displaced by one, two and three pixels.



In [ ]:
shift_flat <- function(X, dx, dy) t(apply(X, 1, function(v) as.vector(shift_img(matrix(v,SG,SG), dx, dy))))
set.seed(20); Gte <- make_glyphs(40)                       # fresh held-out glyphs
raw_mlp <- nnet(Gs, class.ind(G$y), size=16, softmax=TRUE, maxit=250, trace=FALSE, decay=1e-3, MaxNWts=1e5)
shifts <- list(c(0,0),c(1,0),c(0,1),c(1,1),c(2,0),c(2,2),c(3,0))
acc_raw <- sapply(shifts, function(s) mean(predict(raw_mlp, scale(shift_flat(Gte$X,s[1],s[2]),Gctr,Gsd), type="class")==Gte$y))
cat("test accuracy when the glyphs are shifted by (dx,dy):\n")
cat(paste(sapply(seq_along(shifts), function(i) sprintf("(%d,%d):%.3f", shifts[[i]][1], shifts[[i]][2], acc_raw[i])), collapse="  "), "\n")
k <- which(predict(raw_mlp, Gs, type="class")==G$y & G$y=="cross")[1]
par(mfrow=c(1,6), mar=c(1,1,2.4,1))
for (i in 1:5){ s <- shifts[[i]]; im <- shift_img(matrix(G$X[k,],SG,SG), s[1], s[2])
  pr <- predict(raw_mlp, scale(matrix(as.vector(im),1), Gctr, Gsd), type="class")
  image(t(im[SG:1,]), col=grey.colors(32,1,0), axes=FALSE); title(sprintf("shift (%d,%d)\nsaid %s", s[1],s[2],substr(pr,1,4)), cex.main=0.8); box() }
barplot(acc_raw, names.arg=sapply(shifts,function(s)paste0("(",s[1],",",s[2],")")), col=palette_neg,
        ylim=c(0,1), las=2, cex.names=0.6, main="accuracy vs shift", cex.main=0.8)



<p class="caption"><b>Figure 14.</b> Left: a held-out glyph displaced by up to two pixels, with the perceptron's prediction under each. Right: test accuracy as a function of the shift. The unshifted accuracy is high; a displacement of <em>one pixel</em> — one eighth of the image — already lowers it, and two or three pixels drive it towards chance. Nothing about the glyphs changed; only their position did. A radiograph is never positioned to the pixel, and a lesion can appear anywhere in the field of view, so this fragility would be disqualifying in practice. Section 2.3 offers one remedy, augmentation with shifted copies; Section 6.4 measures how far it goes and introduces the second remedy, which is architectural.</p>

### 6.2 Convolution, by hand

A **convolution** slides a small matrix of weights — the **kernel** or **filter**, typically 3 × 3 — over the image and, at each position, outputs the sum of the products of the kernel entries with the pixels under them:

<div class="eq">\[ (\mathbf{I} \star \mathbf{K})_{r,c} = \sum_{u=1}^{k} \sum_{v=1}^{k} K_{u,v}\, I_{r+u-1,\, c+v-1} \]
<span class="where">The output is a <strong>feature map</strong>: large where the image locally resembles the kernel. At the image border the missing neighbours are filled by zero padding ("same" output size) or dropped ("valid", smaller output). Strictly this operation is a <em>cross-correlation</em>; a mathematical convolution flips the kernel first. Deep-learning libraries compute the cross-correlation and call it convolution, and since the kernels are learned the distinction is immaterial. Without padding, a \( k \times k \) kernel on an \( H \times W \) image gives an \( (H-k+1) \times (W-k+1) \) map.</span></div>

The same nine weights are applied at every position: the kernel is a *single* template shared across the image, in contrast to the 64 independent weights per unit of the perceptron. That weight sharing is what gives the operation **translation equivariance** — shift the image and the feature map shifts with it, unchanged otherwise — and the pooling step that follows converts equivariance into (local) invariance. Take a 4 × 4 image with a vertical edge in its upper half and the vertical-edge kernel \( \mathbf{K} \):

\[ \mathbf{I} = \begin{pmatrix} 1 & 1 & 5 & 5 \\ 1 & 1 & 5 & 5 \\ 1 & 1 & 1 & 1 \\ 1 & 1 & 1 & 1 \end{pmatrix}, \qquad \mathbf{K} = \begin{pmatrix} -1 & 0 & 1 \\ -1 & 0 & 1 \\ -1 & 0 & 1 \end{pmatrix} \]

The valid output is 2 × 2. At position (1,1) the window is the top-left 3 × 3 block; each of its rows contributes \( -1 \cdot (\text{left}) + 0 + 1 \cdot (\text{right}) \): rows 1 and 2 give \( -1 + 5 = 4 \) each, row 3 gives \( -1 + 1 = 0 \), total \( 8 \). At (1,2) the window moves one column right: rows 1 and 2 give \( -1 + 5 = 4 \), row 3 gives \( 0 \), total \( 8 \). At (2,1) the window covers rows 2–4: row 2 gives \( 4 \), rows 3 and 4 give \( 0 \), total \( 4 \); at (2,2) likewise \( 4 \). So \( \mathbf{I} \star \mathbf{K} = \left(\begin{smallmatrix} 8 & 8 \\ 4 & 4 \end{smallmatrix}\right) \). With the transposed kernel (a horizontal-edge detector) position (1,1) gives \( -(1+1+5) + (1+1+1) = -4 \) and position (1,2) gives \( -(1+5+5) + 3 = -8 \); the second row repeats them. A **ReLU** sets the negative responses to zero, so this image excites the vertical-edge map and not the horizontal one; a **2 × 2 max-pooling** of the vertical map returns the single number 8 — "there is a vertical edge somewhere in this image", regardless of where. The chunk confirms the numbers.



In [ ]:
conv_valid <- function(I, K){ k <- nrow(K); H <- nrow(I); W <- ncol(I)
  outer(1:(H-k+1), 1:(W-k+1), Vectorize(function(r,c) sum(K * I[r:(r+k-1), c:(c+k-1)]))) }
maxpool <- function(M, s=2){ H <- nrow(M)/s; W <- ncol(M)/s
  outer(1:H,1:W, Vectorize(function(r,c) max(M[((r-1)*s+1):(r*s),((c-1)*s+1):(c*s)]))) }
Iv <- matrix(c(1,1,5,5, 1,1,5,5, 1,1,1,1, 1,1,1,1), 4, 4, byrow=TRUE)
Kv <- matrix(c(-1,0,1, -1,0,1, -1,0,1), 3, 3, byrow=TRUE)
mv <- conv_valid(Iv, Kv); mh <- conv_valid(Iv, t(Kv))
cat("vertical-edge map:\n"); print(mv); cat("horizontal-edge map:\n"); print(mh)
cat("after ReLU + 2x2 max-pool -> vertical:", as.vector(maxpool(pmax(mv,0))),
    " horizontal:", as.vector(maxpool(pmax(mh,0))), "\n")



The code reproduces the hand result — \( \left(\begin{smallmatrix} 8 & 8 \\ 4 & 4 \end{smallmatrix}\right) \) for the vertical kernel and \( \left(\begin{smallmatrix} -4 & -8 \\ -4 & -8 \end{smallmatrix}\right) \) for the horizontal one. After the ReLU and the pooling, the image is summarised by two numbers, 8 and 0: "a vertical edge, no horizontal edge".

### 6.3 Feature maps and pooling on real images

The chunk applies a small **bank of fixed filters** — vertical and horizontal Sobel edge detectors, a Laplacian (responds to points and thin lines), a Gaussian blur, and a difference-of-Gaussians "blob" detector tuned to the size of the simulated nodules — to a glyph and to a nodule patch, with "same" padding, and shows the maps before and after 4 × 4 max-pooling.



In [ ]:
FILTERS <- list(
  `Sobel v`=matrix(c(-1,0,1,-2,0,2,-1,0,1),3,3,byrow=TRUE),
  `Sobel h`=matrix(c(-1,-2,-1,0,0,0,1,2,1),3,3,byrow=TRUE),
  Laplacian=matrix(c(0,-1,0,-1,4,-1,0,-1,0),3,3,byrow=TRUE),
  blur=gkern(1.0,3), blob=gkern(1.2,9)-gkern(2.4,9))
feature_maps <- function(img) lapply(FILTERS, function(K) pmax(conv_same(img,K),0))
set.seed(7); gl <- draw_glyph("diagL"); pt <- nod$X[which(nod$y==1)[1],,]
par(mfrow=c(4,6), mar=c(0.4,0.4,1.4,0.4), oma=c(0,0,2,0))
for (ex in list(list(gl,"glyph"), list(pt,"nodule patch"))){ maps <- feature_maps(ex[[1]])
  image(t(ex[[1]][nrow(ex[[1]]):1,]),col=grey.colors(32,1,0),axes=FALSE,main=ex[[2]],cex.main=0.85); box()   # row: original + maps
  for (nm in names(FILTERS)){ M <- maps[[nm]]; image(t(M[nrow(M):1,]),col=hcl.colors(32,"Inferno"),axes=FALSE,main=nm,cex.main=0.85); box() }
  plot.new()                                                                                                  # row: blank + pooled maps
  for (nm in names(FILTERS)){ Mp <- maxpool(maps[[nm]],4); image(t(Mp[nrow(Mp):1,]),col=hcl.colors(32,"Inferno"),axes=FALSE,main="pooled",cex.main=0.8); box() } }
mtext("Feature maps (ReLU of the correlation) and, below each, its 4x4 max-pool", outer=TRUE, line=0.3, cex=0.9, font=2)
cat("map sizes: patch 16x16 -> pooled", paste(dim(maxpool(feature_maps(pt)[[1]],4)), collapse=" x "), "\n")



<p class="caption"><b>Figure 15.</b> Columns 1 and (per block) the originals; the remaining columns are the five feature maps and, beneath each, its 4 × 4 max-pooled version, for a glyph (first two rows) and a nodule patch (last two rows). The Sobel maps light up on edges according to their orientation; the Laplacian responds to thin structures and noise; the blur map is a smoothed copy; the blob detector responds strongly and selectively to the nodule and weakly to the background texture — it is a template of "a bright round object about this size" applied at every position. Pooling discards the exact position within each 4 × 4 block while keeping the information that the feature was present: a nodule displaced by one or two pixels lands in the same pooled cell. Stacking convolution, non-linearity and pooling several times, with the kernels <em>learned</em> by backpropagation rather than fixed, is the convolutional neural network of LeCun and colleagues,<sup class="c"><a href="#r14">[14]</a></sup> and the one that won ImageNet in 2012.<sup class="c"><a href="#r16">[16]</a></sup></p>

### 6.4 A minimal convolutional front end, and the return of shift robustness

The chunk assembles the pieces into a **fixed-filter convolutional front end**: the filter bank, a ReLU, 4 × 4 max-pooling, and the per-map global maximum and mean, followed by a perceptron on the resulting vector. Nothing in the front end is learned — the filters are the hand-designed ones above — so this is not a convolutional network in the full sense; it is a demonstration that the *structure* of convolution and pooling, not only the learning of the filters, is what buys robustness. The experiment repeats the shift test for four systems: the raw perceptron; the raw perceptron trained with shift augmentation; the convolutional front end; and the front end with augmentation.



In [ ]:
conv_features <- function(X, pool=4){ t(apply(X, 1, function(v){ maps <- feature_maps(matrix(v,SG,SG))
  c(unlist(lapply(maps, function(M) as.vector(maxpool(M,pool)))),
    sapply(maps, max), sapply(maps, mean)) })) }
aug_shifts <- list(c(1,0),c(-1,0),c(0,1),c(0,-1))
Xaug <- rbind(G$X, do.call(rbind, lapply(aug_shifts, function(s) shift_flat(G$X,s[1],s[2])))); yaug <- rep(G$y, 5)
fit_eval <- function(Xtr,ytr,featfun){ Z <- featfun(Xtr); ce <- colMeans(Z); s <- apply(Z,2,sd); s[s==0] <- 1
  fit <- nnet(scale(Z,ce,s), class.ind(ytr), size=16, softmax=TRUE, maxit=200, trace=FALSE, decay=1e-3, MaxNWts=1e5)
  function(Xte) mean(predict(fit, scale(featfun(Xte),ce,s), type="class")==Gte$y) }
idf <- function(X) X
systems <- list(`raw pixels`=fit_eval(G$X,G$y,idf),
                `raw + shift aug`=fit_eval(Xaug,yaug,idf),
                `conv front end`=fit_eval(G$X,G$y,conv_features),
                `conv + aug`=fit_eval(Xaug,yaug,conv_features))
tab <- sapply(systems, function(ev) sapply(shifts[c(1,2,5,7)], function(s) round(ev(shift_flat(Gte$X,s[1],s[2])),3)))
rownames(tab) <- sapply(shifts[c(1,2,5,7)], function(s) sprintf("shift (%d,%d)",s[1],s[2]))
cat("features per image: raw", ncol(G$X), "| conv front end", ncol(conv_features(G$X[1:2,])), "\n")
print(t(tab))



Read the table row by row. The **raw perceptron** falls off quickly with the shift. **Augmentation** repairs mainly the shifts it was shown — one pixel in each direction — and helps less at larger displacements: the network memorised a few extra positions; it did not learn that position is irrelevant. The **convolutional front end** alone, with fixed filters and one pooling stage, is more robust because 4 × 4 pooling tolerates a displacement while the feature stays in the same block. The **combination** is the most robust at the larger shifts: the pooled representation makes a shifted glyph *look similar* to the network, so the invariance learned from small examples extrapolates to shifts it never saw. That extrapolation is what an **inductive bias** is — an assumption built into the architecture that lets the model generalise beyond the cases in the sample — and it is why convolutional networks learned from thousands of radiographs what perceptrons could not learn from millions of pixels.

<div class="box key"><h4>From this front end to deep learning, and to transfer learning</h4><p>A real convolutional network differs from the chunk above in three ways. Its filters are <em>learned</em> by backpropagation together with the classifier, so the blob detector we tuned by hand is discovered from data — and so are detectors nobody thought to design. It stacks several convolution–ReLU–pooling blocks, so that later layers combine edges into parts and parts into objects. And it is trained on far more images than a hospital has — which is why <strong>transfer learning</strong> is the standard recipe in medical imaging: a network pre-trained on the 1.2 million natural images of ImageNet<sup class="c"><a href="#r16">[16]</a></sup> has its last layer replaced and is fine-tuned on a few thousand medical images. Shin and colleagues showed that filters learned on photographs transfer usefully to CT, because edges, blobs and textures are universal;<sup class="c"><a href="#r17">[17]</a></sup> every landmark paper of Section 8 used it. This notebook does not run a deep network, because the course uses no deep-learning framework; MedMNIST<sup class="c"><a href="#r8">[8]</a></sup> with a framework such as Keras or Torch is the place to do so afterwards.</p></div>

### 6.5 The same lesson on the simulated patches

The patches allow a complementary experiment that the glyphs cannot: the **position of the lesion** is a parameter of the generator. The chunk generates three nodule datasets in which the lesion centre is jittered by up to 1, 3 and 6 pixels around the centre of the 16 × 16 patch, and cross-validates the same perceptron on three representations — raw pixels, the 14 hand-crafted features of Section 3, and the convolutional front end with the blob detector.



In [ ]:
conv_feat_patch <- function(X, pool=4){ t(apply(X, 1, function(v){ img <- matrix(v,SIZE,SIZE)
  maps <- lapply(FILTERS, function(K) pmax(conv_same(img,K),0))
  c(sapply(maps, max), sapply(maps, mean), unlist(lapply(maps, function(M) as.vector(maxpool(M,pool))))) })) }
jitters <- c(1,3,6); reps <- list()
for (j in jitters){ set.seed(20+j); pj <- make_patches(500, "nodule", jitter=j)
  Pp <- flat_patch(pj$X); Fp <- t(sapply(1:500, function(k) extract_features(pj$X[k,,]))); Cp <- conv_feat_patch(pj$X)
  fo <- strat_folds(pj$y, 3)
  for (rep in c("raw pixels","14 features","conv front end")){ Xin <- switch(rep, `raw pixels`=Pp, `14 features`=Fp, Cp)
    p <- mlp_bin_cv(Xin, pj$y, 4, fo, 0.5)
    a <- sapply(1:3, function(f){ te <- fo==f; mean((p[te]>0.5)==pj$y[te]) })
    reps[[rep]] <- rbind(reps[[rep]], c(mean(a), sd(a))) } }
cols <- c(palette_neg, palette_acc, palette_pos)
plot(NA, xlim=range(jitters), ylim=c(0.5,1), xlab="max lesion displacement (pixels)", ylab="CV accuracy",
     main="Nodule task: three representations vs lesion position"); i <- 1
for (rep in names(reps)){ m <- reps[[rep]]; lines(jitters, m[,1], type="b", pch=19, col=cols[i])
  arrows(jitters, m[,1]-m[,2], jitters, m[,1]+m[,2], angle=90, code=3, length=0.03, col=cols[i]); i <- i+1 }
legend("bottomleft", names(reps), col=cols, lwd=2, pch=19, bty="n", cex=0.8)
for (rep in names(reps)) cat(sprintf("%-16s %s\n", rep, paste(sprintf("%.3f+/-%.3f", reps[[rep]][,1], reps[[rep]][,2]), collapse="  ")))



<p class="caption"><b>Figure 16.</b> Cross-validated accuracy of the same perceptron on three representations of the nodule patches, as the lesion is allowed to wander further from the centre. On raw pixels the network degrades as the nodule can be anywhere: it has to learn a separate template for every position, and five hundred images are not enough. The hand-crafted features are invariant by construction — area, contrast and texture do not depend on where the blob is — and their curve is the flattest. The convolutional front end sits between the two: its blob detector is applied everywhere and its pooling discards position within 4 × 4 blocks, so it degrades gracefully; a network with learned filters and more pooling stages would be flatter still. The figure is the whole argument of the section in one picture: with little data, invariance must come from somewhere other than the data — from designed features or from the architecture.</p>

<div class="exercise"><h4>Exercise 4</h4><p>Change <code>pool=4</code> to <code>pool=16</code> in <code>conv_feat_patch</code>, so that every map is reduced to a single global maximum and mean. The representation is now fully translation-invariant; is it still discriminative? Then run the microcalcification set (<code>mc</code>) through the same three representations. Which filter of the bank carries the signal for that task, and why do the first-order features fail on it?</p></div>

## 7. What can go wrong in medical image classification

### 7.1 Shortcut learning: the network learns the hospital

Unit 1 simulated a scanner identifier that predicted the label in the training hospital and not elsewhere. With images the confound need not be a separate column: it can be *in the pixels*. Zech and colleagues trained a convolutional network to detect pneumonia on radiographs pooled from several hospitals and found that it identified the source hospital from the image with near-perfect accuracy — from laterality markers, burnt-in text, the field of view — and used the hospital's pneumonia prevalence as a predictor, so that its external performance was far below its internal cross-validation.<sup class="c"><a href="#r18">[18]</a></sup> Geirhos and colleagues named the phenomenon **shortcut learning**: the model finds the easiest regularity that predicts the label in the training distribution, intended or not.<sup class="c"><a href="#r19">[19]</a></sup> DeGrave and colleagues showed that the COVID-19 radiograph detectors of 2020 — positives from one set of sources, negatives from another — relied on markers, image borders and shoulder position rather than on the lungs.<sup class="c"><a href="#r20">[20]</a></sup> The chunk reproduces the mechanism with hospital A, whose positive patches carry a corner marker 90 % of the time, and evaluates the perceptron on new patches from the same hospital and from a hospital B without markers.



In [ ]:
fit_pix <- function(X, y){ ce <- colMeans(X); s <- apply(X,2,sd); s[s==0] <- 1
  m <- nnet(scale(X,ce,s), y, size=6, entropy=TRUE, maxit=250, trace=FALSE, decay=0.5, MaxNWts=1e5)
  function(Xn) as.vector(predict(m, scale(Xn,ce,s))) }
set.seed(10); XA <- make_patches(1000, "nodule", hospital="A", marker_rate=0.9)   # training hospital
set.seed(11); XA2 <- make_patches(400, "nodule", hospital="A", marker_rate=0.9)   # new cases, same hospital
set.seed(12); XB <- make_patches(400, "nodule", hospital="B")                     # another hospital, no markers
set.seed(13); XH <- make_patches(1000, "nodule", hospital="B")                    # a marker-free training set
mA <- fit_pix(flat_patch(XA$X), XA$y); mH <- fit_pix(flat_patch(XH$X), XH$y)
ev <- function(m, d) mean((m(flat_patch(d$X))>0.5)==d$y)
res <- rbind(`trained at hospital A (marker)` = c(`training`=ev(mA,XA), `new, same hospital`=ev(mA,XA2), `new, hospital B`=ev(mA,XB)),
             `trained without markers`         = c(ev(mH,XH), ev(mH,XA2), ev(mH,XB)))
cat(sprintf("fraction of hospital-A positives carrying the marker: %.2f\n",
            mean(apply(XA$X[XA$y==1,,], 1, function(im) all(im[2:3,(SIZE-3):(SIZE-1)]>0.9)))))
print(round(res,3))



The model trained at hospital A looks excellent at home — its accuracy on new cases from the same hospital is well above that of the marker-free model, because the marker is a far easier cue than a faint nodule — and loses many points at hospital B, where it must rely on whatever it learned about nodules, which is less than the marker-free model learned. As in Unit 1, no internal number warned of the failure: training accuracy and same-hospital accuracy were both high. The cause can be made visible, which is the subject of the next section.

### 7.2 Where is the network looking? Occlusion sensitivity

A **saliency map** shows which parts of an image drive a prediction. The simplest, introduced by Zeiler and Fergus, is **occlusion sensitivity**: cover a small square of the image with a neutral value, record how much the predicted probability of the class drops, move the square, and repeat; the map of drops shows where the evidence is.<sup class="c"><a href="#r21">[21]</a></sup> It requires nothing but the ability to call the model's probability output, so it works for a perceptron as well as for a deep network. The drop is measured in log-odds, \( \log[p/(1-p)] \), rather than in probability, because a confident model outputs \( p \approx 1 \) for many patches and a drop from 0.999 to 0.99 — a ten-fold change in the odds — would be invisible on the probability scale. The chunk computes it for the two models of Section 7.1 on three positive hospital-A patches that carry the marker.



In [ ]:
logit <- function(p){ p <- pmin(pmax(p,1e-6),1-1e-6); log(p/(1-p)) }
occ_map <- function(model, img, size=5, step=2){ base <- model(matrix(as.vector(t(img)),1))
  H <- matrix(0,SIZE,SIZE); N <- matrix(0,SIZE,SIZE)
  for (r in seq(1,SIZE-size+1,step)) for (c in seq(1,SIZE-size+1,step)){ oc <- img; oc[r:(r+size-1),c:(c+size-1)] <- mean(img)
    d <- logit(base) - logit(model(matrix(as.vector(t(oc)),1)))
    H[r:(r+size-1),c:(c+size-1)] <- H[r:(r+size-1),c:(c+size-1)] + d; N[r:(r+size-1),c:(c+size-1)] <- N[r:(r+size-1),c:(c+size-1)] + 1 }
  list(map=H/pmax(N,1), p=base) }
idx <- which(XA2$y==1 & apply(XA2$X, 1, function(im) im[2,SIZE-2]>0.9))[1:3]
par(mfrow=c(3,3), mar=c(1,1,2.6,1), oma=c(0,0,2.4,0))
for (i in idx){ om_s <- occ_map(mA, XA2$X[i,,]); om_h <- occ_map(mH, XA2$X[i,,])
  image(t(XA2$X[i,,][SIZE:1,]), col=grey.colors(32,1,0), axes=FALSE, zlim=c(0,1))
  title(sprintf("p=%.2f (A-model)\n%.2f (marker-free)", om_s$p, om_h$p), cex.main=0.85); box()
  for (om in list(list(om_s,"hospital-A model"), list(om_h,"marker-free model"))){ v <- max(abs(om[[1]]$map))+1e-9
    image(t(om[[1]]$map[SIZE:1,]), col=colorRampPalette(c(palette_neg,"white",palette_pos))(32), axes=FALSE, zlim=c(-v,v))
    title(sprintf("%s\nmax drop %.1f", om[[2]], max(om[[1]]$map)), cex.main=0.85); box() } }
mtext("Occlusion sensitivity (orange: covering this region lowers the log-odds of 'nodule')", outer=TRUE, line=0.4, cex=0.9, font=2)



<p class="caption"><b>Figure 17.</b> Three positive patches from hospital A (left) and the occlusion-sensitivity maps of the two models. For the model trained at hospital A (middle) the evidence is the corner marker: covering the top-right corner collapses the probability, and covering the nodule changes little. For the marker-free model (right) the evidence is the nodule itself. The maps are coarse — the occluder is 5 × 5 pixels moved in steps of 2 — but they would have revealed the shortcut before deployment, which is how the markers and borders of the COVID-19 detectors were found.<sup class="c"><a href="#r20">[20]</a></sup> Saliency maps have limits of their own: they show <em>where</em> the model looked, not <em>why</em>, they can be noisy, and a map that highlights the lesion does not prove that the model has understood it. Used as a debugging tool, as here, they are indispensable.</p>

<div class="box warn"><h4>The shortcut need not be a marker</h4><p>Any variable that correlates with the label in the training sample and is visible in the pixels can be a shortcut: the scanner model (through noise texture), the department (portable radiographs are taken on sicker patients), the presence of a chest drain in pneumothorax images — a case documented by Oakden-Rayner and colleagues, whose network detected the treatment rather than the condition<sup class="c"><a href="#r22">[22]</a></sup> — or the ruler placed next to lesions that dermatologists already suspect. The defence is the one Unit 1 gave: evaluate on data from a different site, stratify the evaluation by every variable that could be a shortcut, and look at saliency maps of the errors <em>and</em> of the confident successes.</p></div>

### 7.3 Class imbalance, prevalence and label noise

Two properties of real imaging datasets that the balanced simulations hide deserve an experiment each. **Class imbalance**: in screening, positives are rare, and a classifier trained on 95 % negatives learns that "negative" is a good answer; its accuracy is high and its sensitivity is low. The simplest remedy is often to keep the model and **move the threshold**, as Section 4.5 did. **Label noise**: the labels of the large radiograph datasets were extracted from reports by text mining and are wrong in a known fraction of cases;<sup class="c"><a href="#r3">[3]</a></sup> the survey of Frénay and Verleysen reviews what noise does to classifiers.<sup class="c"><a href="#r23">[23]</a></sup> Two things happen: the model learns worse, and — more insidiously — the *measured* accuracy on a noisily labelled test set is capped below the true accuracy, so a perfect model looks imperfect.



In [ ]:
set.seed(30); Xneg <- make_patches(950, "nodule", positive=FALSE); set.seed(31); Xpos <- make_patches(50, "nodule", positive=TRUE)
Xi <- rbind(flat_patch(Xneg$X), flat_patch(Xpos$X)); Fi <- rbind(t(sapply(1:950,function(k)extract_features(Xneg$X[k,,]))),
                                                                 t(sapply(1:50,function(k)extract_features(Xpos$X[k,,]))))
yi <- c(rep(0,950), rep(1,50)); set.seed(0); foi <- strat_folds(yi, 5)
pi_oof <- mlp_bin_cv(Fi, yi, 6, foi, 0.5)
roc_i <- roc(yi, pi_oof, quiet=TRUE, direction="<")
t90 <- coords(roc_i, x=0.90, input="sensitivity", ret="threshold")[1,1]
metrics <- function(pred) c(accuracy=mean(pred==yi), sensitivity=mean(pred[yi==1]), specificity=1-mean(pred[yi==0]), `positives called`=sum(pred))
res_i <- rbind(`always negative`=metrics(rep(0,length(yi))), `MLP, threshold 0.5`=metrics(as.integer(pi_oof>=0.5)),
               `MLP, threshold for 90% sens`=metrics(as.integer(pi_oof>=t90)))
cat(sprintf("prevalence %.1f%% | ROC-AUC %.3f | threshold for 90%% sensitivity %.2f\n", 100*mean(yi), as.numeric(auc(roc_i)), t90))
print(round(res_i,3))



The trivial rule that calls everything negative has an accuracy of 0.95 — the prevalence of negatives — and a sensitivity of zero. The perceptron on the hand-crafted features does better than that at the default threshold and still misses a share of the nodules, because with 50 positives among 1,000 the cheapest way to reduce the training loss is to be conservative; the ROC-AUC shows that its probabilities rank the cases well. Lowering the threshold to the value that reaches 90 % sensitivity recovers most of the missed cases at the cost of more positive calls and a *lower* accuracy. On an imbalanced task accuracy is not a measure of anything useful (Supplement S1); sensitivity, specificity, the ROC curve and, for the clinician, the PPV at the deployment prevalence are.



In [ ]:
set.seed(40); tr <- sample(nrow(G$X), round(0.7*nrow(G$X))); levs <- GLYPHS
flip_labels <- function(y, q){ y <- as.character(y); fl <- runif(length(y))<q
  y[fl] <- sapply(which(fl), function(i) sample(setdiff(levs, y[i]), 1)); factor(y, levels=levs) }
qs <- c(0,0.1,0.2,0.3,0.4); clean_acc <- noisy_acc <- numeric(length(qs))
for (k in seq_along(qs)){ set.seed(40+k); ytr <- flip_labels(G$y[tr], qs[k]); yte_n <- flip_labels(G$y[-tr], qs[k])
  fit <- nnet(Gs[tr,], class.ind(ytr), size=24, softmax=TRUE, maxit=200, trace=FALSE, decay=1e-3, MaxNWts=1e5)
  pr <- predict(fit, Gs[-tr,], type="class"); clean_acc[k] <- mean(pr==G$y[-tr]); noisy_acc[k] <- mean(pr==yte_n) }
plot(qs, clean_acc, type="b", pch=19, col=palette_neg, ylim=c(0.3,1.02), xlab="fraction of flipped training labels",
     ylab="test accuracy", main="Label noise on the glyphs")
lines(qs, noisy_acc, type="b", pch=15, lty=2, col=palette_pos); lines(qs, 1-qs, lty=3, col="grey40")
legend("bottomleft", c("vs CLEAN labels","vs equally NOISY labels","ceiling 1 - q"),
       col=c(palette_neg,palette_pos,"grey40"), lty=c(1,2,3), pch=c(19,15,NA), bty="n", cex=0.75)
cat("clean-label accuracy:", round(clean_acc,3), "| noisy-label accuracy:", round(noisy_acc,3), "\n")



<p class="caption"><b>Figure 18.</b> Test accuracy of the glyph perceptron as the fraction of flipped training labels grows, measured against clean test labels (blue) and against test labels corrupted at the same rate (orange). The blue curve degrades gradually, because wrong labels spread over the other classes partly cancel out; with more training images the curve would be flatter. The orange curve is the one a researcher with noisy labels would actually see: it runs below the grey ceiling \( 1-q \) by about the model's true error, so a model that is in fact accurate is <em>reported</em> as far worse at high noise. Noise that is <em>systematic</em> — a report-mining rule that always misses one kind of pneumonia — is not averaged out and is far more damaging; the uncertainty labels of CheXpert<sup class="c"><a href="#r4">[4]</a></sup> and the radiologist-adjudicated test sets of the landmark papers exist because of this.</p>

### 7.4 External validation, reader studies and the regulatory status of imaging AI

Three practices separate a model that works in a notebook from a device that works in a hospital. **External validation** evaluates the frozen model on data from institutions, scanners and periods that contributed nothing to its development; the CLAIM checklist for AI in medical imaging makes it a reporting requirement, together with the sample, the reference standard, the pre-processing, the model-selection procedure and metrics with confidence intervals.<sup class="c"><a href="#r24">[24]</a></sup> A **reader study** compares the model with clinicians reading the same cases under the same conditions, ideally also reading *with* the model, because the deployed system is the combination. **Prospective evaluation** measures what happens to patients when the model is used.

The regulatory record shows how rarely these are done. Benjamens and colleagues catalogued 64 AI-based devices cleared or approved by the United States Food and Drug Administration by 2020, the majority in radiology, and found that most public summaries disclosed little about the evaluation.<sup class="c"><a href="#r25">[25]</a></sup> Wu and colleagues examined 130 such devices: 126 had been evaluated only retrospectively, 93 reported no multi-site evaluation, and few reported performance by demographic subgroup.<sup class="c"><a href="#r26">[26]</a></sup> Most are **software as a medical device** cleared through the 510(k) pathway by equivalence to an existing device, not through clinical trials; the framework of Unit 1 — the European AI Act's high-risk classification, the FDA's good-machine-learning-practice principles, the WHO's six principles — is the context in which the engineer's choices in this unit become legal obligations.

## 8. Reading the landmark papers

The papers below defined the field between 2016 and 2020 and are the ones your Workshop 2 presentations are most likely to cite. For each, the questions of Unit 1 are asked: what were the objects, the labels and their reference standard, the sample, the metric, the comparator — and what is missing. The table is a reading guide, not a substitute for the papers.

| Paper | Task and data | Headline result | Comparator | What to ask |
|---|---|---|---|---|
| Gulshan et al., <em>JAMA</em> 2016<sup class="c"><a href="#r27">[27]</a></sup> | Referable diabetic retinopathy on 128,175 retinal photographs, each graded by several of 54 ophthalmologists; validated on two sets of 9,963 and 1,748 images | AUC about 0.99 on both; at the high-sensitivity operating point, sensitivity about 97 % and specificity about 93 % | Majority vote of ophthalmologists as reference standard | Retrospective; images from the cameras and populations of the training sites; ungradeable images excluded; no patient outcome |
| Esteva et al., <em>Nature</em> 2017<sup class="c"><a href="#r28">[28]</a></sup> | Skin lesions: a network pre-trained on ImageNet and fine-tuned on 129,450 clinical images of 2,032 diseases; tested on biopsy-proven images | Sensitivity–specificity at or above the operating points of 21 dermatologists on three binary tasks | 21 board-certified dermatologists reading the same images | Dermatologists read photographs, not patients; the test images were selected; skin tones under-represented |
| Rajpurkar et al., 2017 (CheXNet)<sup class="c"><a href="#r29">[29]</a></sup> | Pneumonia on ChestX-ray14; 121-layer convolutional network; test set of 420 images labelled by four radiologists | F1 score above the average of the four radiologists | Radiologists reading the same frontal images without clinical history | Training labels mined from reports with known errors;<sup class="c"><a href="#r3">[3]</a></sup> single institution; wide confidence intervals on 420 images |
| McKinney et al., <em>Nature</em> 2020<sup class="c"><a href="#r30">[30]</a></sup> | Breast-cancer screening mammography: 25,856 women in the UK and 3,097 in the US, with cancer status from follow-up | Fewer false positives and negatives than the original readers; outperformed six radiologists in a reader study | Historical first-reader decisions; a reader study of six radiologists | Haibe-Kains et al. replied that code, model and data were not shared, so the result cannot be reproduced or audited;<sup class="c"><a href="#r31">[31]</a></sup> retrospective |
| Ardila et al., <em>Nat Med</em> 2019<sup class="c"><a href="#r32">[32]</a></sup> | Lung-cancer screening on low-dose CT from the National Lung Screening Trial: 3-D network on whole volumes; cancer confirmed by biopsy | AUC about 0.94 on 6,716 cases; fewer false positives and negatives than six radiologists when no prior CT was available | Six radiologists, with and without the prior scan | Trial data of a single country and era; small independent validation; no prospective use |
| Liu et al., <em>Lancet Digit Health</em> 2019<sup class="c"><a href="#r33">[33]</a></sup> | Systematic review of deep learning versus health-care professionals on medical images: 82 of more than 30,000 studies met the inclusion criteria | Pooled sensitivity about 87 % versus 86 % and specificity about 93 % versus 91 % on the few studies with a fair comparison | Clinicians, in the 14 studies that compared both on the same external sample | Only 25 of 82 studies validated externally; few reported confidence intervals; the evidence base is poor |

Three patterns run through the table. The reference standard is a human judgement or a biopsy, never the model's own training labels, and the most credible papers used several readers and an adjudication. The comparator is almost always radiologists reading images *without* the clinical context they would have in practice, which favours the machine. And none of the original papers was prospective: the strongest claim any supports is "on retrospective images from these sources, the model's operating point is comparable to these readers", which is where Topol's 2019 review placed the evidence.<sup class="c"><a href="#r34">[34]</a></sup> The vocabulary of this course — sample, label, leakage, external validation, prevalence, confidence interval, shortcut — is sufficient to read every one of these papers critically.

<div class="box bridge"><h4>Bridge to Unit 9</h4><p>Unit 9 returns to these systems from the side of deployment: how a classifier becomes a diagnostic system with a user interface, a workflow and a monitoring plan; what data mining of the electronic record adds to imaging; and the professional-ethics questions that Sections 7.1–7.4 raised in technical form. The experiments of this unit — the topology grid, the shift test, the shortcut and its occlusion map — are the evidence you will be asked to produce in that discussion.</p></div>

## 9. Summary

You should now be able to explain, without notes:

- What a pixel value encodes in radiography, CT (Hounsfield units), MRI, ultrasound, histopathology, dermoscopy and retinal photography; why CT values are comparable across scanners and MRI values are not; what windowing, bit depth and DICOM are.
- Why an image is a matrix and a feature vector at once, what flattening discards (adjacency), and the five stages of the imaging pipeline.
- What min–max scaling, standardisation, histogram equalisation (and the algorithm behind it), resizing, Gaussian denoising and augmentation do, and the label-preservation condition that governs augmentation.
- How to compute first-order, GLCM texture and shape features; how to trace a co-occurrence matrix and its contrast, homogeneity and energy by hand; why designed features beat raw pixels with a thousand images.
- How a perceptron classifies flattened images, how to estimate its accuracy with repeated stratified cross-validation, how to read its confusion matrix and its first-layer templates, and how to choose a threshold for a screening prevalence.
- The four rules of an honest topology experiment, what the depth × width grid, the learning curves and the penalty/activation/optimiser experiments showed, and why nested cross-validation is needed to report the result.
- Why a perceptron is not translation-invariant, how to trace a convolution, a ReLU and a max-pooling by hand, and why convolution plus pooling plus augmentation generalises to shifts that augmentation alone does not.
- What shortcut learning is, how to reproduce it with a hospital marker, how an occlusion-sensitivity map reveals it, and why external validation, stratified evaluation and saliency inspection are the defences.
- Why accuracy is meaningless under class imbalance, what label noise does to a model and to its measured accuracy, and what the regulatory record says about how imaging AI has been evaluated.
- How to read a landmark paper: objects, labels, reference standard, sample, metric, comparator, and what is missing.

<div class="box warn"><h4>What is simulated here, and what the Python edition did differently</h4><p>R bundles no handwritten-digit or medical-image dataset, so this unit builds both of its image sources from code: a 6-class set of 8 × 8 glyphs in place of the Python edition's handwritten digits, and 16 × 16 radiograph-like patches in place of its 32 × 32 patches. The numbers therefore differ from the Python notebook, but every qualitative conclusion is the same. One language difference shapes Section 5: R's <code>nnet</code> offers a single logistic hidden layer optimised by BFGS, so depth was studied with <code>neuralnet</code> (on a PCA representation, for convergence) and the learning-rate/momentum/batch experiment was run on a from-scratch gradient-descent trainer with a rectified-linear activation. The union of these three tools covers the ground the single Python <code>MLPClassifier</code> covered.</p></div>

## 10. Self-assessment

<details><summary>1. A CT voxel has the value −850 and an MRI voxel has the value 850. What can you say about the tissue in each case, and why is the answer different in kind?</summary>The CT value is in Hounsfield units, a physical scale: −850 is close to air, so the voxel is most likely aerated lung. The MRI value is in arbitrary units that depend on the scanner, coil and pulse sequence; by itself it says nothing, and only its relation to other voxels of the same acquisition is informative. This is why MRI requires per-image intensity normalisation and CT uses windowing.</details>

<details><summary>2. Trace the histogram equalisation of a 2 × 2 image with values (0, 0, 1, 3) on a 4-level scale.</summary>Histogram: level 0 occurs twice, level 1 once, level 2 never, level 3 once. Cumulative distribution F: F(0) = 2/4 = 0.5, F(1) = 0.75, F(2) = 0.75, F(3) = 1. New value in [0,1] is F itself (or rescaled to the level range). The two dark pixels map to 0.5, the 1 maps to 0.75 and the 3 to 1: the image's values are pushed up and spread, and the contrast between the two dark pixels disappears — equalisation of a tiny image is coarse.</details>

<details><summary>3. In the augmentation chunk, which of the six transformations preserves the label of a chest radiograph classified for pneumonia? For a mammogram classified for laterality (left/right breast)? For a dermoscopic image classified for melanoma?</summary>Pneumonia: all six preserve the label, though left–right flipping should be avoided if the model will also report the side. Laterality: the left–right flip destroys the label (it is the label), the others preserve it. Melanoma: all six, because a skin lesion has no canonical orientation or exposure.</details>

<details><summary>4. Compute the GLCM contrast of the 3 × 3 image [[0,1,0],[0,1,0],[0,1,0]] with 2 levels and the displacement "one pixel to the right".</summary>Horizontal pairs: each row gives (0,1) and (1,0), six pairs in all, all with |i − j| = 1. P(0,1) = P(1,0) = 1/2; contrast = Σ (i − j)² P = 1 · (1/2 + 1/2) = 1. Homogeneity = Σ P/(1 + |i − j|) = 1/2; energy = 1/4 + 1/4 = 1/2. A vertical stripe pattern has maximal horizontal contrast.</details>

<details><summary>5. The perceptron of Section 4.1 reported an accuracy of (say) 0.93 ± 0.02 over ten splits. A colleague's network scores 0.95 on one 70/30 split of the same data. Is it better?</summary>Not demonstrably. The ten splits of our model ranged over several percentage points, so 0.95 is within the spread of a single split of the same model. A fair comparison trains both on the same repeated splits and compares the paired differences across folds; and if the colleague selected the architecture on that split, the 0.95 is optimistically biased (Section 5.6).</details>

<details><summary>6. In the depth × width grid the decisive jump was from the linear model to one hidden layer, while adding more layers changed the accuracy only within the fold-to-fold noise. Why should extra depth give so little reliable gain at this sample size?</summary>A single hidden layer already makes the classes separable, so it captures most of the available signal and lifts accuracy far above the linear baseline. Each additional layer adds capacity, but with only a few hundred images that capacity cannot be constrained by the data, so any gain is small and uncertain (comparable to the standard deviation across folds) and the optimisation is harder because the gradient must pass through more layers. "Deep" helps when the data are large enough to constrain the extra parameters; with a small sample the experiment cannot demonstrate a depth benefit.</details>

<details><summary>7. Why is the best cell of a cross-validated grid not an unbiased estimate of the chosen model's performance, and what estimate should be reported instead?</summary>The best cell was chosen because it was the maximum of several noisy estimates on the same folds; the maximum of noisy quantities is biased upwards. The nested cross-validation estimate — the outer-fold accuracy of the procedure "choose by inner CV, then fit" — is unbiased for the deployed procedure and is what should be reported.</details>

<details><summary>8. Compute by hand the valid cross-correlation of [[2,2,2],[2,2,2],[0,0,0]] with the kernel [[1,1,1],[0,0,0],[−1,−1,−1]], then the result after ReLU.</summary>The kernel is 3 × 3 and the image 3 × 3, so the output is a single number: 1·(2+2+2) + 0·(2+2+2) − 1·(0+0+0) = 6. ReLU(6) = 6. The kernel detects "bright above dark" and the image is exactly that, so the response is maximal. With the kernel flipped vertically the response would be −6 and the ReLU would give 0.</details>

<details><summary>9. In the shift table of Section 6.4, augmentation with one-pixel shifts repaired small shifts but not larger ones, whereas the convolutional front end with augmentation was robust to both. Explain the difference in one sentence each.</summary>Augmentation taught the raw perceptron a few additional positions by rote, and a larger shift is a position it never saw; the pooled convolutional representation makes shifted images <em>similar as vectors</em>, so what the network learned from small examples transfers to larger shifts — the architecture supplies an inductive bias that the data alone did not.</details>

<details><summary>10. The model trained at hospital A had higher same-hospital accuracy than the marker-free model, yet it is the worse model. Which single evaluation would have shown this, and which diagnostic would have shown why?</summary>Evaluation on patches from a different hospital without the marker (external validation) showed the drop; the occlusion-sensitivity map showed that the evidence used was the corner marker rather than the nodule.</details>

<details><summary>11. A classifier for a 5 % prevalence task reports 95 % accuracy. What is the least it could be doing, and which two numbers would you ask for?</summary>Calling every case negative achieves 95 % accuracy with zero sensitivity. Ask for sensitivity and specificity (or the ROC curve), and then compute the PPV at the deployment prevalence; accuracy is not informative under imbalance.</details>

<details><summary>12. In the label-noise experiment, which curve would a researcher with noisily labelled data observe, and what does it imply for comparing a model against radiologists on such a test set?</summary>The orange curve, measured against noisy test labels, which is capped near 1 − q. A model can appear far worse than it is, and a comparison with radiologists on the same noisy labels is distorted in a direction that depends on whether the radiologists' errors correlate with the label noise. The remedy is a test set with an adjudicated reference standard, as the landmark papers used.</details>

## Glossary

<dl class="gloss">
<dt>Augmentation</dt><dd>Enlarging the training sample with transformed copies of its images (flips, rotations, shifts, intensity changes) under the condition that the transformation preserves the label; applied to training folds only.</dd>
<dt>Convolution (cross-correlation)</dt><dd>Sliding a small kernel over an image and recording, at each position, the sum of the products of kernel weights and underlying pixels; the output is a feature map. Deep-learning libraries compute the cross-correlation and call it convolution.</dd>
<dt>Convolutional neural network (CNN)</dt><dd>A network whose early layers are learned convolutions followed by non-linearities and pooling, so that the same feature detectors are applied at every position; the architecture of LeCun (1998) and of AlexNet (2012).</dd>
<dt>DICOM</dt><dd>Digital Imaging and Communications in Medicine: the standard file and communication format of medical images, whose header stores acquisition parameters and patient data.</dd>
<dt>Equivariance / invariance (translation)</dt><dd>A representation is equivariant if shifting the input shifts the representation correspondingly, and invariant if the representation does not change; convolution is equivariant, pooling makes it locally invariant.</dd>
<dt>Feature map</dt><dd>The output of one kernel applied across the whole image; large where the image locally resembles the kernel.</dd>
<dt>Grey-level co-occurrence matrix (GLCM)</dt><dd>Haralick's texture descriptor: the normalised counts of pairs of grey levels at a given displacement, from which contrast, homogeneity, energy and correlation are computed.</dd>
<dt>Histogram equalisation</dt><dd>Non-linear intensity mapping through the cumulative distribution of the image's histogram, which spreads the grey levels over the available range.</dd>
<dt>Hounsfield unit (HU)</dt><dd>The CT intensity scale: 1000 (μ − μ<sub>water</sub>)/(μ<sub>water</sub> − μ<sub>air</sub>), so that water is 0 and air is −1000; a physical quantity comparable across scanners.</dd>
<dt>Inductive bias</dt><dd>The assumptions built into a model's architecture or training that let it generalise beyond the exact cases in the sample; convolution and pooling encode the assumption that the class does not depend on position.</dd>
<dt>Label noise</dt><dd>Errors in the training or test labels; uniform noise degrades a model mildly but caps the measured accuracy at about 1 − q; systematic noise is learned as if it were signal.</dd>
<dt>Learning curve</dt><dd>Cross-validated and training accuracy plotted against the number of training examples; the diagnostic for deciding whether more data or a different model is needed.</dd>
<dt>Nested cross-validation</dt><dd>An outer cross-validation that estimates the performance of a procedure in which the model configuration is chosen by an inner cross-validation on the training folds only.</dd>
<dt>Occlusion sensitivity</dt><dd>A saliency map obtained by covering successive regions of the image and recording the drop in the predicted probability of the class.</dd>
<dt>Pooling (max-pooling)</dt><dd>Replacing each block of a feature map by its maximum (or mean), which reduces the map's size and discards the exact position of the feature within the block.</dd>
<dt>Radiomics</dt><dd>The computation of large standardised sets of first-order, texture and shape features from segmented lesions, followed by classical machine learning.</dd>
<dt>Reader study</dt><dd>An evaluation in which clinicians and the model read the same cases under comparable conditions; the standard comparator in the landmark papers.</dd>
<dt>Shortcut learning</dt><dd>A model's reliance on a spurious regularity of the training data — a marker, a border, a scanner texture — that predicts the label there but not elsewhere.</dd>
<dt>Template (first-layer weights)</dt><dd>The weight vector of a hidden unit reshaped to the image size; it shows the pixel pattern to which the unit responds most strongly.</dd>
<dt>Transfer learning</dt><dd>Reusing a network pre-trained on a large dataset (ImageNet) by replacing its last layer and fine-tuning on a smaller medical dataset.</dd>
<dt>Windowing</dt><dd>Mapping a chosen interval of CT values to the displayed grey scale, clipping values outside it; different windows show lung, soft tissue or bone on the same slice.</dd>
</dl>

## References and further reading

<ol class="refs">
<li id="r1">Litjens G, Kooi T, Bejnordi BE, et al. A survey on deep learning in medical image analysis. <em>Med Image Anal</em>. 2017;42:60–88. <a href="https://doi.org/10.1016/j.media.2017.07.005">doi:10.1016/j.media.2017.07.005</a></li>
<li id="r2">Hounsfield GN. Computerized transverse axial scanning (tomography): Part 1. Description of system. <em>Br J Radiol</em>. 1973;46(552):1016–1022. <a href="https://doi.org/10.1259/0007-1285-46-552-1016">doi:10.1259/0007-1285-46-552-1016</a></li>
<li id="r3">Wang X, Peng Y, Lu L, Lu Z, Bagheri M, Summers RM. ChestX-ray8: Hospital-scale chest X-ray database and benchmarks on weakly-supervised classification and localization of common thorax diseases. <em>Proc IEEE CVPR</em>. 2017:3462–3471. <a href="https://doi.org/10.1109/CVPR.2017.369">doi:10.1109/CVPR.2017.369</a></li>
<li id="r4">Irvin J, Rajpurkar P, Ko M, et al. CheXpert: a large chest radiograph dataset with uncertainty labels and expert comparison. <em>Proc AAAI Conf Artif Intell</em>. 2019;33(1):590–597. <a href="https://doi.org/10.1609/aaai.v33i01.3301590">doi:10.1609/aaai.v33i01.3301590</a></li>
<li id="r5">Johnson AEW, Pollard TJ, Berkowitz SJ, et al. MIMIC-CXR, a de-identified publicly available database of chest radiographs with free-text reports. <em>Sci Data</em>. 2019;6:317. <a href="https://doi.org/10.1038/s41597-019-0322-0">doi:10.1038/s41597-019-0322-0</a></li>
<li id="r6">Lee RS, Gimenez F, Hoogi A, Miyake KK, Gorovoy M, Rubin DL. A curated mammography data set for use in computer-aided detection and diagnosis research. <em>Sci Data</em>. 2017;4:170177. <a href="https://doi.org/10.1038/sdata.2017.177">doi:10.1038/sdata.2017.177</a></li>
<li id="r7">Tschandl P, Rosendahl C, Kittler H. The HAM10000 dataset, a large collection of multi-source dermatoscopic images of common pigmented skin lesions. <em>Sci Data</em>. 2018;5:180161. <a href="https://doi.org/10.1038/sdata.2018.161">doi:10.1038/sdata.2018.161</a></li>
<li id="r8">Yang J, Shi R, Wei D, et al. MedMNIST v2 — a large-scale lightweight benchmark for 2D and 3D biomedical image classification. <em>Sci Data</em>. 2023;10:41. <a href="https://doi.org/10.1038/s41597-022-01721-8">doi:10.1038/s41597-022-01721-8</a></li>
<li id="r9">Street WN, Wolberg WH, Mangasarian OL. Nuclear feature extraction for breast tumor diagnosis. <em>Proc SPIE</em>. 1993;1905:861–870. <a href="https://doi.org/10.1117/12.148698">doi:10.1117/12.148698</a></li>
<li id="r10">Alpaydin E, Kaynak C. Optical Recognition of Handwritten Digits. UCI Machine Learning Repository; 1998. <a href="https://doi.org/10.24432/C50P49">doi:10.24432/C50P49</a></li>
<li id="r11">Setio AAA, Traverso A, de Bel T, et al. Validation, comparison, and combination of algorithms for automatic detection of pulmonary nodules in computed tomography images: the LUNA16 challenge. <em>Med Image Anal</em>. 2017;42:1–13. <a href="https://doi.org/10.1016/j.media.2017.06.015">doi:10.1016/j.media.2017.06.015</a></li>
<li id="r12">Shorten C, Khoshgoftaar TM. A survey on image data augmentation for deep learning. <em>J Big Data</em>. 2019;6:60. <a href="https://doi.org/10.1186/s40537-019-0197-0">doi:10.1186/s40537-019-0197-0</a></li>
<li id="r13">Haralick RM, Shanmugam K, Dinstein I. Textural features for image classification. <em>IEEE Trans Syst Man Cybern</em>. 1973;SMC-3(6):610–621. <a href="https://doi.org/10.1109/TSMC.1973.4309314">doi:10.1109/TSMC.1973.4309314</a></li>
<li id="r14">LeCun Y, Bottou L, Bengio Y, Haffner P. Gradient-based learning applied to document recognition. <em>Proc IEEE</em>. 1998;86(11):2278–2324. <a href="https://doi.org/10.1109/5.726791">doi:10.1109/5.726791</a></li>
<li id="r15">Varma S, Simon R. Bias in error estimation when using cross-validation for model selection. <em>BMC Bioinformatics</em>. 2006;7:91. <a href="https://doi.org/10.1186/1471-2105-7-91">doi:10.1186/1471-2105-7-91</a></li>
<li id="r16">Krizhevsky A, Sutskever I, Hinton GE. ImageNet classification with deep convolutional neural networks. <em>Commun ACM</em>. 2017;60(6):84–90. <a href="https://doi.org/10.1145/3065386">doi:10.1145/3065386</a></li>
<li id="r17">Shin H-C, Roth HR, Gao M, et al. Deep convolutional neural networks for computer-aided detection: CNN architectures, dataset characteristics and transfer learning. <em>IEEE Trans Med Imaging</em>. 2016;35(5):1285–1298. <a href="https://doi.org/10.1109/TMI.2016.2528162">doi:10.1109/TMI.2016.2528162</a></li>
<li id="r18">Zech JR, Badgeley MA, Liu M, Costa AB, Titano JJ, Oermann EK. Variable generalization performance of a deep learning model to detect pneumonia in chest radiographs: a cross-sectional study. <em>PLoS Med</em>. 2018;15(11):e1002683. <a href="https://doi.org/10.1371/journal.pmed.1002683">doi:10.1371/journal.pmed.1002683</a></li>
<li id="r19">Geirhos R, Jacobsen J-H, Michaelis C, et al. Shortcut learning in deep neural networks. <em>Nat Mach Intell</em>. 2020;2:665–673. <a href="https://doi.org/10.1038/s42256-020-00257-z">doi:10.1038/s42256-020-00257-z</a></li>
<li id="r20">DeGrave AJ, Janizek JD, Lee S-I. AI for radiographic COVID-19 detection selects shortcuts over signal. <em>Nat Mach Intell</em>. 2021;3:610–619. <a href="https://doi.org/10.1038/s42256-021-00338-7">doi:10.1038/s42256-021-00338-7</a></li>
<li id="r21">Zeiler MD, Fergus R. Visualizing and understanding convolutional networks. In: <em>Computer Vision — ECCV 2014</em>. Lecture Notes in Computer Science, vol 8689. Springer; 2014:818–833. <a href="https://doi.org/10.1007/978-3-319-10590-1_53">doi:10.1007/978-3-319-10590-1_53</a></li>
<li id="r22">Oakden-Rayner L, Dunnmon J, Carneiro G, Ré C. Hidden stratification causes clinically meaningful failures in machine learning for medical imaging. <em>Proc ACM Conf Health Inference Learn (CHIL)</em>. 2020:151–159. <a href="https://doi.org/10.1145/3368555.3384468">doi:10.1145/3368555.3384468</a></li>
<li id="r23">Frénay B, Verleysen M. Classification in the presence of label noise: a survey. <em>IEEE Trans Neural Netw Learn Syst</em>. 2014;25(5):845–869. <a href="https://doi.org/10.1109/TNNLS.2013.2292894">doi:10.1109/TNNLS.2013.2292894</a></li>
<li id="r24">Mongan J, Moy L, Kahn CE Jr. Checklist for Artificial Intelligence in Medical Imaging (CLAIM): a guide for authors and reviewers. <em>Radiol Artif Intell</em>. 2020;2(2):e200029. <a href="https://doi.org/10.1148/ryai.2020200029">doi:10.1148/ryai.2020200029</a></li>
<li id="r25">Benjamens S, Dhunnoo P, Meskó B. The state of artificial intelligence-based FDA-approved medical devices and algorithms: an online database. <em>npj Digit Med</em>. 2020;3:118. <a href="https://doi.org/10.1038/s41746-020-00324-0">doi:10.1038/s41746-020-00324-0</a></li>
<li id="r26">Wu E, Wu K, Daneshjou R, Ouyang D, Ho DE, Zou J. How medical AI devices are evaluated: limitations and recommendations from an analysis of FDA approvals. <em>Nat Med</em>. 2021;27:582–584. <a href="https://doi.org/10.1038/s41591-021-01312-x">doi:10.1038/s41591-021-01312-x</a></li>
<li id="r27">Gulshan V, Peng L, Coram M, et al. Development and validation of a deep learning algorithm for detection of diabetic retinopathy in retinal fundus photographs. <em>JAMA</em>. 2016;316(22):2402–2410. <a href="https://doi.org/10.1001/jama.2016.17216">doi:10.1001/jama.2016.17216</a></li>
<li id="r28">Esteva A, Kuprel B, Novoa RA, et al. Dermatologist-level classification of skin cancer with deep neural networks. <em>Nature</em>. 2017;542:115–118. <a href="https://doi.org/10.1038/nature21056">doi:10.1038/nature21056</a></li>
<li id="r29">Rajpurkar P, Irvin J, Zhu K, et al. CheXNet: radiologist-level pneumonia detection on chest X-rays with deep learning. <em>arXiv</em>. 2017;1711.05225. <a href="https://doi.org/10.48550/arXiv.1711.05225">doi:10.48550/arXiv.1711.05225</a> (preprint; not peer-reviewed)</li>
<li id="r30">McKinney SM, Sieniek M, Godbole V, et al. International evaluation of an AI system for breast cancer screening. <em>Nature</em>. 2020;577:89–94. <a href="https://doi.org/10.1038/s41586-019-1799-6">doi:10.1038/s41586-019-1799-6</a></li>
<li id="r31">Haibe-Kains B, Adam GA, Hosny A, et al. Transparency and reproducibility in artificial intelligence. <em>Nature</em>. 2020;586:E14–E16. <a href="https://doi.org/10.1038/s41586-020-2766-y">doi:10.1038/s41586-020-2766-y</a></li>
<li id="r32">Ardila D, Kiraly AP, Bharadwaj S, et al. End-to-end lung cancer screening with three-dimensional deep learning on low-dose chest computed tomography. <em>Nat Med</em>. 2019;25:954–961. <a href="https://doi.org/10.1038/s41591-019-0447-x">doi:10.1038/s41591-019-0447-x</a></li>
<li id="r33">Liu X, Faes L, Kale AU, et al. A comparison of deep learning performance against health-care professionals in detecting diseases from medical imaging: a systematic review and meta-analysis. <em>Lancet Digit Health</em>. 2019;1(6):e271–e297. <a href="https://doi.org/10.1016/S2589-7500(19)30123-2">doi:10.1016/S2589-7500(19)30123-2</a></li>
<li id="r34">Topol EJ. High-performance medicine: the convergence of human and artificial intelligence. <em>Nat Med</em>. 2019;25:44–56. <a href="https://doi.org/10.1038/s41591-018-0300-7">doi:10.1038/s41591-018-0300-7</a></li>
<li id="r35">Venables WN, Ripley BD. <em>Modern Applied Statistics with S</em>. 4th ed. Springer; 2002. <a href="https://doi.org/10.1007/978-0-387-21706-2">doi:10.1007/978-0-387-21706-2</a> (the reference for the <code>nnet</code> single-hidden-layer perceptron used here)</li>
<li id="r36">Fritsch S, Guenther F, Wright MN. neuralnet: Training of Neural Networks. R package. <a href="https://doi.org/10.32614/CRAN.package.neuralnet">doi:10.32614/CRAN.package.neuralnet</a></li>
<li id="r37">Meyer D, Dimitriadou E, Hornik K, Weingessel A, Leisch F. e1071: Misc Functions of the Department of Statistics, Probability Theory Group (Formerly E1071), TU Wien. R package. <a href="https://doi.org/10.32614/CRAN.package.e1071">doi:10.32614/CRAN.package.e1071</a></li>
<li id="r38">Liaw A, Wiener M. randomForest: Breiman and Cutler's random forests for classification and regression. R package. <a href="https://doi.org/10.32614/CRAN.package.randomForest">doi:10.32614/CRAN.package.randomForest</a></li>
<li id="r39">Robin X, Turck N, Hainard A, et al. pROC: an open-source package for R and S+ to analyze and compare ROC curves. <em>BMC Bioinformatics</em>. 2011;12:77. <a href="https://doi.org/10.1186/1471-2105-12-77">doi:10.1186/1471-2105-12-77</a></li>
</ol>

<h4>Textbooks</h4>
<ul>
<li>James G, Witten D, Hastie T, Tibshirani R. <em>An Introduction to Statistical Learning with Applications in R</em>. 2nd ed. Springer; 2021. — The R companion to the whole course; free PDF at statlearning.com.</li>
<li>Goodfellow I, Bengio Y, Courville A. <em>Deep Learning</em>. MIT Press; 2016. — Chapter 9 (convolutional networks) is the reference for Section 6; Chapter 7 for regularisation and early stopping; Chapter 8 for the optimisers of Section 5.5. Free online at deeplearningbook.org.</li>
<li>Bishop CM. <em>Pattern Recognition and Machine Learning</em>. Springer; 2006. — Chapter 5 for the multilayer perceptron, including the invariance and convolution discussion.</li>
<li>Hastie T, Tibshirani R, Friedman J. <em>The Elements of Statistical Learning</em>. 2nd ed. Springer; 2009. — Chapter 7 for model assessment and the bias of model selection; Chapter 11 for neural networks. Free PDF.</li>
<li>Gonzalez RC, Woods RE. <em>Digital Image Processing</em>. 4th ed. Pearson; 2018. — Chapters 3 and 10 for histogram equalisation, filtering and segmentation.</li>
</ul>

<h4>Online resources</h4>
<ul>
<li>CRAN Task View: Machine Learning &amp; Statistical Learning — <a href="https://cran.r-project.org/view=MachineLearning">cran.r-project.org/view=MachineLearning</a>; the catalogue of R packages for the methods of this unit.</li>
<li>MedMNIST — <a href="https://medmnist.com">medmnist.com</a>; small, standardised medical image datasets; the recommended next step after this notebook (with a deep-learning framework).</li>
<li>ISIC Archive — <a href="https://www.isic-archive.com">isic-archive.com</a>; public dermoscopic images and the yearly melanoma classification challenges.</li>
<li>DICOM standard — <a href="https://www.dicomstandard.org">dicomstandard.org</a>; the <code>oro.dicom</code> R package reads DICOM files.</li>
<li>CLAIM checklist — <a href="https://pubs.rsna.org/page/ai/claim">pubs.rsna.org/page/ai/claim</a>; the reporting checklist of Section 7.4 in its current version.</li>
</ul>
